In [ ]:
import pandas as pd

# Hardcoded paths
file1 = "all_59k_accidents_raw.csv"
file2 = "weather_59k_merged.csv"
output_file = "all_59k_accidents_weather.csv"

# Column names
lat_col_1 = "latitude"
lon_col_1 = "longitude"
lat_col_2 = "source_latitude"
lon_col_2 = "source_longitude"

# Read as strings for exact matching
df1 = pd.read_csv(file1, dtype=str)
df2 = pd.read_csv(file2, dtype=str)

# Strip whitespace from column names
df1.columns = df1.columns.str.strip()
df2.columns = df2.columns.str.strip()

# Validate required columns
required_1 = {lat_col_1, lon_col_1}
required_2 = {lat_col_2, lon_col_2}

missing_1 = required_1 - set(df1.columns)
missing_2 = required_2 - set(df2.columns)

if missing_1:
    raise ValueError(f"First file missing columns: {sorted(missing_1)}")
if missing_2:
    raise ValueError(f"Second file missing columns: {sorted(missing_2)}")

# Strip whitespace from coordinate values
df1[lat_col_1] = df1[lat_col_1].str.strip()
df1[lon_col_1] = df1[lon_col_1].str.strip()

df2[lat_col_2] = df2[lat_col_2].str.strip()
df2[lon_col_2] = df2[lon_col_2].str.strip()

# Select columns to append: everything after source_longitude
second_cols = list(df2.columns)
start_idx = second_cols.index(lon_col_2) + 1
append_cols = second_cols[start_idx:]

if not append_cols:
    raise ValueError(f"No columns found after '{lon_col_2}' in file2.")

# Create lookup table
df2_small = df2[[lat_col_2, lon_col_2] + append_cols].copy()

# Add explicit match flag
df2_small["_matched"] = True

# Remove duplicate coordinate pairs
dup_count = df2_small.duplicated(
    subset=[lat_col_2, lon_col_2]
).sum()

if dup_count > 0:
    print(f"Warning: {dup_count} duplicate coordinate pairs found in file2. Keeping first occurrence.")
    df2_small = df2_small.drop_duplicates(
        subset=[lat_col_2, lon_col_2],
        keep="first"
    )

# Merge on exact latitude/longitude pair
merged = df1.merge(
    df2_small,
    left_on=[lat_col_1, lon_col_1],
    right_on=[lat_col_2, lon_col_2],
    how="left"
)

# Match statistics
unmatched_rows = merged["_matched"].isna().sum()
matched_rows = len(merged) - unmatched_rows

print(f"Total rows in file1: {len(df1):,}")
print(f"Matched rows:        {matched_rows:,}")
print(f"Unmatched rows:      {unmatched_rows:,}")

# Drop helper and duplicate join columns from file2
merged = merged.drop(columns=["_matched", lat_col_2, lon_col_2])

# Save output
merged.to_csv(output_file, index=False)

print(f"Saved merged file to: {output_file}")

In [ ]:
import pandas as pd
from google.colab import files

# Upload file
uploaded = files.upload()

input_csv = "final_processed_with_interest_a_g_e_final.csv"
output_csv = "updated_out.csv"

df = pd.read_csv(input_csv)

if "Casualty G" not in df.columns:
    raise KeyError("Column 'Casualty G' not found.")

# Work on lowercase text
col = df["Casualty G"].astype(str).str.lower().str.strip()

# Match whole words only
female_mask = col.str.contains(r"\bfemale\b", regex=True, na=False)
male_mask = col.str.contains(r"\bmale\b", regex=True, na=False) & ~female_mask

# Replace values
df.loc[female_mask, "Casualty G"] = 2
df.loc[male_mask, "Casualty G"] = 1

# Save
df.to_csv(output_csv, index=False)

print(f"Saved: {output_csv}")
print(f"Replaced female with 2: {int(female_mask.sum())}")
print(f"Replaced male with 1: {int(male_mask.sum())}")

In [ ]:
import pandas as pd
from google.colab import files


# Replace with your filename
input_csv = "updated_output_5.csv"
output_csv = "updated_output_6.csv"

# Load data
df = pd.read_csv(input_csv)

# Clean column names just in case
df.columns = df.columns.str.strip()

if "weather_conditions" not in df.columns:
    raise KeyError("Column 'weather_conditions' not found.")

# Convert to numeric so it catches values like -1, -1.0, 9, 99
road_type_num = pd.to_numeric(df["weather_conditions"], errors="coerce")

# Mask for values to empty
mask = road_type_num.isin([-1, 9, 99])

# Count and empty
emptied_count = int(mask.sum())
df.loc[mask, "weather_conditions"] = pd.NA

# Save result
df.to_csv(output_csv, index=False)

print(f"Saved: {output_csv}")
print(f"Cells emptied in 'weather_conditions': {emptied_count}")
print("Value counts of emptied values:")
print(road_type_num[mask].value_counts(dropna=False).sort_index())

In [ ]:
import pandas as pd
from google.colab import files

# Upload file


# Replace with your filename
input_csv = "final_processed_with_interest_a_g_e_final.csv"
output_csv = "cleaned_output.csv"

# Load data
df = pd.read_csv(input_csv)

# Normalize column names
df.columns = df.columns.str.strip()

targets = {-1, 99}
matching_columns = []

for col in df.columns:
    # Try numeric comparison first
    numeric_vals = pd.to_numeric(df[col], errors="coerce")
    has_target = numeric_vals.isin(targets).any()

    # Also catch string versions like "-1", "9", "99"
    if not has_target:
        text_vals = df[col].astype(str).str.strip()
        has_target = text_vals.isin({"-1", "99"}).any()

    if has_target:
        matching_columns.append(col)

print(f"Columns with at least one value of -1, or 99 ({len(matching_columns)}):")
for col in matching_columns:
    print(f"- {col}")

In [ ]:
import pandas as pd
from google.colab import files



# Replace with your filename
input_csv = "cleaned_output_2.csv"
output_csv = "updated_output.csv"

# Load data
df = pd.read_csv(input_csv)

if "Casualty G" not in df.columns:
    raise KeyError("The CSV must contain a 'Casualty G' column.")

# Normalize values in Casualty G
original = df["Casualty G"].copy()

def map_gender(value):
    if pd.isna(value):
        return value
    text = str(value).strip().lower()
    if text == "male":
        return 1
    if text == "female":
        return 2
    if text == "":
        return pd.NA
    return value  # leave anything else unchanged

df["Casualty G"] = df["Casualty G"].apply(map_gender)

# Find other values that were not male/female/blank
unexpected_mask = (
    original.notna()
    & (original.astype(str).str.strip() != "")
    & (~original.astype(str).str.strip().str.lower().isin(["male", "female"]))
)

unexpected_rows = df.loc[unexpected_mask, "Casualty G"]

# Save updated file
df.to_csv(output_csv, index=False)

print(f"Saved: {output_csv}")
print(f"Rows with unexpected values in Casualty G: {unexpected_mask.sum()}")

if unexpected_mask.sum() > 0:
    print("\nUnexpected values found in Casualty G:")
    print(original[unexpected_mask].value_counts(dropna=False))



In [ ]:
import pandas as pd
from google.colab import files



# Replace with your filename
input_csv = "final_processed_with_interest_a_g_e_final.csv"
output_csv = "cleaned_output_2.csv"

# Load data
df = pd.read_csv(input_csv)

# Count replacements
replaced_count = 0

def should_empty(cell):
    if pd.isna(cell):
        return False

    text = str(cell).strip().lower()

    # Empty if it contains "unknown" anywhere
    if "unknown" in text:
        return True

    # Empty if it is exactly -1
    if text == "-1":
        return True

    # Also handle numeric -1
    try:
        num = float(cell)
        if num == -1:
            return True
    except Exception:
        pass

    return False

# Replace matching cells with empty values
for col in df.columns:
    mask = df[col].apply(should_empty)
    replaced_count += mask.sum()
    df.loc[mask, col] = pd.NA

# Save result
df.to_csv(output_csv, index=False)

print(f"Saved: {output_csv}")
print(f"Cells emptied: {replaced_count}")


In [ ]:
import pandas as pd
from google.colab import files

# Upload file
uploaded = files.upload()

# Replace with your filename
input_csv = "final_processed_with_interest_age_gender_exposure.csv"
output_csv = "dropped_constant_columns.csv"

# Load data
df = pd.read_csv(input_csv)

# Normalize empty strings to NaN so they are treated consistently
df_normalized = df.replace(r"^\s*$", pd.NA, regex=True)

# Find columns with only one unique value across all rows
constant_columns = [
    col for col in df_normalized.columns
    if df_normalized[col].nunique(dropna=False) == 1
]

# Drop them
df_cleaned = df.drop(columns=constant_columns)

# Save result
df_cleaned.to_csv(output_csv, index=False)

print(f"Dropped columns ({len(constant_columns)}):")
for col in constant_columns:
    print(f"- {col}")

print(f"\nSaved cleaned file as: {output_csv}")

# Download
files.download(output_csv)

In [ ]:
import pandas as pd
from google.colab import files



# Replace with your filename
input_csv = "final_processed_with_interest_age_gender_exposure.csv"
output_csv = "cleaned_output.csv"

# Load data
df = pd.read_csv(input_csv)

# Helper: treat NaN and empty strings as empty
def is_empty(series):
    return series.isna() | (series.astype(str).str.strip() == "")

# Identify columns where ALL values are empty
empty_columns = [col for col in df.columns if is_empty(df[col]).all()]

# Drop them
df_cleaned = df.drop(columns=empty_columns)

# Save result
df_cleaned.to_csv(output_csv, index=False)

# Print results
print(f"Dropped columns ({len(empty_columns)}):")
for col in empty_columns:
    print(f"- {col}")

print(f"\nSaved cleaned file as: {output_csv}")


In [ ]:
import pandas as pd
from google.colab import files


# Replace with your filename
input_csv = "final_processed_with_interest_age_gender_exposure.csv"
output_csv = "filtered_empty_rows.csv"

# Load data
df = pd.read_csv(input_csv)

# Helper function to detect empty values
def is_empty(series):
    return series.isna() | (series.astype(str).str.strip() == "")

# Filter rows where BOTH columns are empty
filtered_df = df[is_empty(df["Casualty G"])]

# Save result
filtered_df.to_csv(output_csv, index=False)

print(f"Saved: {output_csv}")
print(f"Rows found: {len(filtered_df)}")


In [ ]:
import pandas as pd
import re
from google.colab import files



# Replace these with your actual uploaded filenames
first_csv = "final_processed_with_interest_age_gender_exposure.csv"
second_csv = "all_29k_accidents_raw.csv"

output_csv = "updated_first.csv"

# Read CSVs
df1 = pd.read_csv(first_csv)
df2 = pd.read_csv(second_csv)

sex_re = re.compile(r"\b(male|female)\b", re.IGNORECASE)

def extract_sex(value):
    if pd.isna(value):
        return None
    match = sex_re.search(str(value))
    return match.group(1).lower() if match else None

def is_empty(value):
    return pd.isna(value) or str(value).strip() == ""

unmatched_count = 0

# Match rows by row order
max_rows = min(len(df1), len(df2))

for i in range(max_rows):
    if is_empty(df1.at[i, "Casualty G"]):
        sex = extract_sex(df2.at[i, "Casualty S"])
        if sex is None:
            unmatched_count += 1
        else:
            df1.at[i, "Casualty G"] = sex

# Count any extra empty rows in df1 beyond df2
if len(df1) > len(df2):
    for i in range(len(df2), len(df1)):
        if is_empty(df1.at[i, "Casualty G"]):
            unmatched_count += 1

# Save result
df1.to_csv(output_csv, index=False)

print(f"Saved: {output_csv}")
print(f"Rows that did not find a usable Casualty S match: {unmatched_count}")

# Download the result
files.download(output_csv)

In [ ]:
import pandas as pd
from collections import defaultdict, deque

# ==== CONFIG ====
csv1_path = "all_29k_accidents_raw.csv"
csv2_path = "final_processed_with_interest.csv"
output_path = "final_processed_with_interest_merged.csv"

lat1_col = "latitude"
lon1_col = "longitude"

lat2_col = "latitude"
lon2_col = "longitude"

casualty_g_col = "Casualty G"   # keep exact spelling from CSV1
casualty_a_col = "Casualty A"

precision = 6  # adjust if needed


# ==== LOAD FILES ====
df1 = pd.read_csv(csv1_path)
df2 = pd.read_csv(csv2_path)

# ==== CLEAN + ROUND ====
def make_df_with_pair(df, lat_col, lon_col, precision):
    df = df[[lat_col, lon_col]].copy()
    df = df.dropna(subset=[lat_col, lon_col])
    df[lat_col] = df[lat_col].astype(float).round(precision)
    df[lon_col] = df[lon_col].astype(float).round(precision)
    df["pair"] = list(zip(df[lat_col], df[lon_col]))
    return df

df1_coords = make_df_with_pair(df1, lat1_col, lon1_col, precision)
df2_coords = make_df_with_pair(df2, lat2_col, lon2_col, precision)

# ==== BUILD LOOKUP FROM CSV1 ====
pair_to_values = defaultdict(deque)

df1_with_vals = df1_coords.join(df1[[casualty_g_col, casualty_a_col]])
for _, row in df1_with_vals.iterrows():
    pair_to_values[row["pair"]].append((row[casualty_g_col], row[casualty_a_col]))

# ==== MATCH CSV2 TO CSV1 AND APPEND COLUMNS ====
df2_result = df2.copy()
df2_result["pair"] = list(zip(
    df2_result[lat2_col].astype(float).round(precision),
    df2_result[lon2_col].astype(float).round(precision)
))

df2_result[casualty_g_col] = pd.NA
df2_result[casualty_a_col] = pd.NA

matched_rows = 0  # number of CSV2 rows that got a match and received values

for i, pair in enumerate(df2_result["pair"]):
    if pair in pair_to_values and len(pair_to_values[pair]) > 0:
        g_val, a_val = pair_to_values[pair].popleft()
        df2_result.at[i, casualty_g_col] = g_val
        df2_result.at[i, casualty_a_col] = a_val
        matched_rows += 1

# ==== MOVE NEW COLUMNS RIGHT AFTER "seconds" ====
cols = list(df2_result.columns)

# remove helper/new columns first
for c in ["pair", casualty_g_col, casualty_a_col]:
    if c in cols:
        cols.remove(c)

if "seconds" in cols:
    seconds_idx = cols.index("seconds")
    cols = cols[:seconds_idx + 1] + [casualty_g_col, casualty_a_col] + cols[seconds_idx + 1:]
    df2_result = df2_result[cols]
else:
    df2_result = df2_result.drop(columns=["pair"], errors="ignore")

# ==== RESULTS ====
print(f"Rows in CSV1 (valid coords): {len(df1_coords)}")
print(f"Rows in CSV2 (valid coords): {len(df2_coords)}")
print(f"Matched rows: {matched_rows}")
print(f"Rows updated with copied columns: {matched_rows}")

# ==== SAVE ====
df2_result.to_csv(output_path, index=False)
print(f"Saved merged file to: {output_path}")

In [ ]:
import pandas as pd

# paths
csv1_path = "all_accidents_raw.csv"
csv2_path = "final_processed_with_interest.csv"

# load
df1 = pd.read_csv(csv1_path)
df2 = pd.read_csv(csv2_path)

# counts
print(f"Rows in CSV1: {len(df1)}")
print(f"Rows in CSV2: {len(df2)}")

In [ ]:
import pandas as pd

files = [
    "2013.csv",
    "2014.csv",
    "2015.csv",
    "2016.csv",
    "2017.csv",
    "2018.csv",
    "2019.csv",
    "2020.csv",
    "2021.csv",
    "2022.csv",
    "2023.csv",
    "2024.csv"
]

df_list = [pd.read_csv(f) for f in files]
merged_df = pd.concat(df_list, ignore_index=True)

merged_df.to_csv("all_29k_accidents_raw.csv", index=False)

print(f"Total rows: {len(merged_df)}")

In [ ]:
import pandas as pd
from collections import Counter

# ==== CONFIG ====
csv1_path = "all_29k_accidents_raw.csv"
csv2_path = "exposure_all_years_tidied.csv"

lat1_col = "latitude"
lon1_col = "longitude"

lat2_col = "latitude"
lon2_col = "longitude"

precision = 6  # adjust if needed


# ==== LOAD FILES ====
df1 = pd.read_csv(csv1_path)
df2 = pd.read_csv(csv2_path)

# ==== CLEAN + ROUND ====
def make_pairs(df, lat_col, lon_col, precision):
    df = df[[lat_col, lon_col]].dropna()
    df[lat_col] = df[lat_col].astype(float).round(precision)
    df[lon_col] = df[lon_col].astype(float).round(precision)
    return list(zip(df[lat_col], df[lon_col]))

pairs1 = make_pairs(df1, lat1_col, lon1_col, precision)
pairs2 = make_pairs(df2, lat2_col, lon2_col, precision)

# ==== MATCH COUNT ====
counter2 = Counter(pairs2)

matched = 0
for pair in pairs1:
    if counter2[pair] > 0:
        matched += 1
        counter2[pair] -= 1  # remove this line if you want "exists in csv2" instead of 1-to-1 match

# ==== RESULTS ====
print(f"Rows in CSV1: {len(pairs1)}")
print(f"Rows in CSV2: {len(pairs2)}")
print(f"Matched rows: {matched}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import BallTree

# ==== CONFIG ====
csv1_path = "exposure_all_years_tidied.csv"
csv2_path = "final_processed_with_interest_age_gender.csv"
output_path = "final_processed_with_interest_age_gender_exposure.csv"

lat1_col = "latitude"
lon1_col = "longitude"

lat2_col = "latitude"
lon2_col = "longitude"

exposure_cyclists = "exposure_avg_cyclists"   # keep exact spelling from CSV1

EARTH_RADIUS = 6371000  # meters


# ==== LOAD FILES ====
df1 = pd.read_csv(csv1_path, low_memory=False)
df2 = pd.read_csv(csv2_path, low_memory=False)

# ==== CLEAN ====
df1_clean = df1[[lat1_col, lon1_col, exposure_cyclists]].copy()
df1_clean = df1_clean.dropna(subset=[lat1_col, lon1_col])
df1_clean[lat1_col] = pd.to_numeric(df1_clean[lat1_col], errors="coerce")
df1_clean[lon1_col] = pd.to_numeric(df1_clean[lon1_col], errors="coerce")
df1_clean = df1_clean.dropna(subset=[lat1_col, lon1_col])

df2_clean = df2[[lat2_col, lon2_col]].copy()
df2_clean[lat2_col] = pd.to_numeric(df2_clean[lat2_col], errors="coerce")
df2_clean[lon2_col] = pd.to_numeric(df2_clean[lon2_col], errors="coerce")
valid_mask = df2_clean[[lat2_col, lon2_col]].notna().all(axis=1)
df2_valid = df2_clean[valid_mask].copy()

# keep original row positions so we can write results back
df2_valid = df2_valid.reset_index().rename(columns={"index": "orig_index"})

print(f"Rows in CSV1 (valid coords): {len(df1_clean)}")
print(f"Rows in CSV2 (valid coords): {len(df2_valid)}")

# ==== NEAREST NEIGHBOR MATCHING ====
coords1 = np.radians(df1_clean[[lat1_col, lon1_col]].values)
coords2 = np.radians(df2_valid[[lat2_col, lon2_col]].values)

tree = BallTree(coords1, metric="haversine")
distances, indices = tree.query(coords2, k=1)

distances_m = distances.flatten() * EARTH_RADIUS
matched_source = df1_clean.iloc[indices.flatten()].reset_index(drop=True)

# ==== BUILD OUTPUT ====
df2_result = df2.copy()
df2_result[exposure_cyclists] = pd.NA

# write matched values back into the original CSV2 rows
df2_result.loc[df2_valid["orig_index"].values, exposure_cyclists] = matched_source[exposure_cyclists].values

# ==== MOVE NEW COLUMN RIGHT AFTER "seconds" ====
cols = list(df2_result.columns)
if exposure_cyclists in cols:
    cols.remove(exposure_cyclists)

if "seconds" in cols:
    seconds_idx = cols.index("seconds")
    cols = cols[:seconds_idx + 1] + [exposure_cyclists] + cols[seconds_idx + 1:]
    df2_result = df2_result[cols]

# ==== RESULTS ====
matched_rows = len(distances_m)
print(f"Matched rows: {matched_rows}")
print(f"Rows updated with copied column: {matched_rows}")
print(f"Min distance (m): {distances_m.min():.3f}")
print(f"Max distance (m): {distances_m.max():.3f}")
print(f"Average distance (m): {distances_m.mean():.3f}")
print(f"Median distance (m): {np.median(distances_m):.3f}")

# ==== HISTOGRAM ====
plt.figure(figsize=(10, 5))
plt.hist(distances_m, bins=30)
plt.title("Nearest-neighbor matching distance distribution")
plt.xlabel("Distance (meters)")
plt.ylabel("Occurrences")
plt.tight_layout()
plt.show()

# Optional: print histogram counts in text form
counts, bin_edges = np.histogram(distances_m, bins=30)
print("\nHistogram bins:")
for i in range(len(counts)):
    print(f"{bin_edges[i]:.2f} - {bin_edges[i+1]:.2f} m: {counts[i]}")

# ==== SAVE ====
df2_result.to_csv(output_path, index=False)
print(f"\nSaved merged file to: {output_path}")

In [ ]:
#!/usr/bin/env python3
"""
Enrich an accident dataset with historical accident-count features.

What it does
------------
1) Reads the source CSV.
2) Deduplicates to one row per collision_id for the counting step.
3) Builds 9 features:
   - radii: 50m, 100m, 200m
   - lookback windows: 1, 2, 3 years
4) Counts how many *previous* accidents happened within each distance/time window.
5) Merges the features back to the original table so every casualty row gets the
   same accident-history features as its collision_id.

Why these windows?
------------------
On this file, these windows give a good spread between:
- very local repetition (50m),
- street/block-level repetition (100m),
- neighborhood-level repetition (200m),

while 1/2/3 years provide short-, medium-, and longer-term history without
becoming too broad.

Output column names
-------------------
past_accidents_50m_1y
past_accidents_50m_2y
past_accidents_50m_3y
past_accidents_100m_1y
...
past_accidents_200m_3y
"""

from __future__ import annotations

import argparse
import math
import re
from collections import defaultdict, deque
from pathlib import Path
from typing import Deque, Dict, Tuple

import numpy as np
import pandas as pd


RADIUS_LIST_M = [50, 100, 200, 300, 400, 500]
YEARS_LIST = [1,2,3, 4, 5]
CELL_SIZE_M = 100.0  # grid cell size for fast local searches


def parse_accident_datetime(df: pd.DataFrame) -> pd.Series:
    """Parse a robust accident datetime from accidentdate + time.

    Falls back to date-only when time is missing or malformed.
    """
    accident_date = pd.to_datetime(df["accidentdate"], errors="coerce", dayfirst=True)

    # Build a combined string where possible, then fallback to date only.
    time_str = df["time"].astype("string").str.strip()
    combined = accident_date.dt.strftime("%Y-%m-%d") + " " + time_str
    dt = pd.to_datetime(combined, errors="coerce")
    dt = dt.fillna(accident_date)
    return dt


def cell_of(x: float, y: float, cell_size: float) -> Tuple[int, int]:
    return (int(math.floor(x / cell_size)), int(math.floor(y / cell_size)))


def count_past_accidents(
    accidents: pd.DataFrame,
    years_list: list[int] = YEARS_LIST,
    radii_list_m: list[int] = RADIUS_LIST_M,
    cell_size_m: float = CELL_SIZE_M,
    severity_list: list[str] = ["Slight", "Serious", "Fatal"],
) -> pd.DataFrame:
    """Count prior accidents within each (year, radius, severity) combination.

    accidents must already be deduplicated to one row per collision_id and
    sorted chronologically by `dt`.
    """
    accidents = accidents.copy()

    max_years = max(years_list)
    max_radius = max(radii_list_m)
    max_radius_sq = max_radius * max_radius
    r_cells = int(math.ceil(max_radius / cell_size_m))

    severity_to_idx = {s: i for i, s in enumerate(severity_list)}

    # Store only recent points needed for the maximum lookback window.
    grid: Dict[Tuple[int, int], Deque[Tuple[pd.Timestamp, float, float, int]]] = defaultdict(deque)

    # All severities combined
    counts = np.zeros(
        (len(accidents), len(years_list), len(radii_list_m)),
        dtype=np.int32,
    )

    # Severity-specific counts
    severity_counts = np.zeros(
        (len(accidents), len(years_list), len(radii_list_m), len(severity_list)),
        dtype=np.int32,
    )

    # Local references for speed.
    years_list = list(years_list)
    radii_sq = [r * r for r in radii_list_m]

    for i, row in enumerate(accidents.itertuples(index=False)):
        dt = row.dt
        x = float(row.easting)
        y = float(row.northing)

        max_cutoff = dt - pd.DateOffset(years=max_years)

        cx, cy = cell_of(x, y, cell_size_m)

        # Scan only nearby grid cells.
        for ix in range(cx - r_cells, cx + r_cells + 1):
            for iy in range(cy - r_cells, cy + r_cells + 1):
                dq = grid.get((ix, iy))
                if not dq:
                    continue

                # Remove points older than the largest lookback window.
                while dq and dq[0][0] < max_cutoff:
                    dq.popleft()

                for past_dt, px, py, past_sev_idx in dq:
                  if past_dt >= dt:
                      continue

                  dx = x - px
                  dy = y - py
                  d2 = dx * dx + dy * dy

                  for yi, years in enumerate(years_list):
                      cutoff = dt - pd.DateOffset(years=years)
                      if past_dt < cutoff:
                          continue

                      for ri, r2 in enumerate(radii_sq):
                          if d2 <= r2:
                              counts[i, yi, ri] += 1
                              severity_counts[i, yi, ri, past_sev_idx] += 1
        # Add the current accident after counting so it cannot count itself.
        current_sev_idx = severity_to_idx.get(row.collision_severity, 0)
        grid[(cx, cy)].append((dt, x, y, current_sev_idx))

    out = accidents[["collision_id"]].copy()
    out = out.set_index("collision_id")

    for yi, years in enumerate(years_list):
        for ri, radius_m in enumerate(radii_list_m):

            # All severities combined
            col = f"past_accidents_{radius_m}m_{years}y"
            out[col] = counts[:, yi, ri]

            # Severity-specific
            for si, sev in enumerate(severity_list):
                sev_col = f"past_accidents_{radius_m}m_{years}y_{sev.lower()}"
                out[sev_col] = severity_counts[:, yi, ri, si]
    out = out.reset_index()
    return out

def main() -> None:
    input_path = "all_59k_accidents_no_history.csv"
    output_path = "all_59k_accidents_complete_final_cleaned.csv"

    df = pd.read_csv(input_path, low_memory=False)

    if "collision_id" not in df.columns:
        raise ValueError("Missing collision_id")
    if "accidentdate" not in df.columns or "time" not in df.columns:
        raise ValueError("Missing accidentdate/time")
    if "easting" not in df.columns or "northing" not in df.columns:
        raise ValueError("Missing coordinates")

    severity_order = {"Slight": 1, "Serious": 2, "Fatal": 3}

    accidents = df[
        ["collision_id", "accidentdate", "time", "easting", "northing", "casualty_severity"]
    ].copy()

    accidents["_sev_rank"] = (
        accidents["casualty_severity"].map(severity_order).fillna(0).astype(int)
    )

    accidents = (
        accidents.sort_values(["collision_id", "_sev_rank"])
        .drop_duplicates("collision_id", keep="last")
        .drop(columns=["_sev_rank"])
    )

    accidents["dt"] = parse_accident_datetime(accidents)
    accidents = accidents.dropna(subset=["dt"])
    accidents = accidents.sort_values(["dt", "collision_id"]).reset_index(drop=True)

    accidents = accidents.rename(columns={"casualty_severity": "collision_severity"})

    features = count_past_accidents(accidents)

    enriched = df.merge(features, on="collision_id", how="left")

    count_cols = [c for c in enriched.columns if c.startswith("past_accidents_")]
    enriched[count_cols] = enriched[count_cols].fillna(0).astype("int32")

    enriched.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")


main()

In [ ]:
#!/usr/bin/env python3
"""
QA checks for accident-history feature engineering.

Run this in Colab or locally after generating the enriched CSV.

Checks performed:
1) File shape and column integrity
2) Missing value checks
3) Collision-level consistency of the 9 new features
4) Monotonic nesting checks
5) Value sanity checks (integer, non-negative)
6) Parsing/timestamp checks
7) Sample brute-force verification of the count logic against the raw data
"""

import numpy as np
import pandas as pd


RAW_PATH = "/content/all_59k_accidents_no_history.csv"
ENRICHED_PATH = "/content/all_59k_accidents_complete_final_cleaned.csv"

COUNT_COLS = [
    "past_accidents_50m_1y",
    "past_accidents_50m_2y",
    "past_accidents_50m_3y",
    "past_accidents_50m_4y",
    "past_accidents_50m_5y",

    "past_accidents_100m_1y",
    "past_accidents_100m_1y_slight",
    "past_accidents_100m_1y_fatal",
    "past_accidents_100m_2y",
    "past_accidents_100m_3y",
    "past_accidents_100m_4y",
    "past_accidents_100m_5y",

    "past_accidents_200m_1y",
    "past_accidents_200m_2y",
    "past_accidents_200m_3y",
    "past_accidents_200m_4y",
    "past_accidents_200m_5y",

    "past_accidents_300m_1y",
    "past_accidents_300m_2y",
    "past_accidents_300m_3y",
    "past_accidents_300m_4y",
    "past_accidents_300m_5y",

    "past_accidents_400m_1y",
    "past_accidents_400m_2y_slight",
    "past_accidents_400m_3y",
    "past_accidents_400m_4y_fatal",
    "past_accidents_400m_5y",

    "past_accidents_500m_1y",
    "past_accidents_500m_2y",
    "past_accidents_500m_3y",
    "past_accidents_500m_4y",
    "past_accidents_500m_5y",

]


def parse_accident_datetime(df: pd.DataFrame) -> pd.Series:
    accident_date = pd.to_datetime(df["accidentdate"], errors="coerce", dayfirst=True)
    time_str = df["time"].astype("string").str.strip()
    combined = accident_date.dt.strftime("%Y-%m-%d") + " " + time_str
    dt = pd.to_datetime(combined, errors="coerce")
    dt = dt.fillna(accident_date)
    return dt


def main() -> None:
    raw = pd.read_csv(RAW_PATH, low_memory=False)
    enr = pd.read_csv(ENRICHED_PATH, low_memory=False)

    print("=== BASIC FILE CHECKS ===")
    print(f"Raw shape:      {raw.shape}")
    print(f"Enriched shape:  {enr.shape}")
    print(f"Added columns:   {sorted(set(enr.columns) - set(raw.columns))}")
    print(f"Removed columns: {sorted(set(raw.columns) - set(enr.columns))}")
    print()

    print("=== NEW COLUMN SANITY ===")
    missing_count_cols = [c for c in COUNT_COLS if c not in enr.columns]
    if missing_count_cols:
        raise ValueError(f"Missing expected count columns: {missing_count_cols}")
    print("All 9 count columns are present.")

    print("\n=== VALUE CHECKS ===")
    count_block = enr[COUNT_COLS]
    print("Null counts:\n", count_block.isna().sum().to_string())
    print("\nDtypes:\n", count_block.dtypes.to_string())
    print("\nMinimums:\n", count_block.min().to_string())
    print("\nMaximums:\n", count_block.max().to_string())
    print("\nAny negatives?", bool((count_block < 0).any().any()))
    print("All integer-like?", bool(np.all(np.mod(count_block.to_numpy(), 1) == 0)))

    print("\n=== COLLISION-LEVEL CONSISTENCY ===")
    nunique_by_collision = enr.groupby("collision_id")[COUNT_COLS].nunique(dropna=False)
    inconsistent = (nunique_by_collision > 1).any(axis=1).sum()
    print(f"Collision IDs with inconsistent feature values across duplicate rows: {int(inconsistent)}")

    print("\n=== MONOTONICITY CHECKS ===")
    checks = []
    for y in [1, 2, 3]:
        checks.append((f"50m <= 100m ({y}y)", bool((enr[f'past_accidents_50m_{y}y'] <= enr[f'past_accidents_100m_{y}y']).all())))
        checks.append((f"100m <= 200m ({y}y)", bool((enr[f'past_accidents_100m_{y}y'] <= enr[f'past_accidents_200m_{y}y']).all())))
    for label, ok in checks:
        print(f"{label}: {ok}")

    print("\n=== TIME PARSING CHECKS ===")
    acc = raw[["collision_id", "accidentdate", "time", "easting", "northing"]].drop_duplicates("collision_id").copy()
    acc["dt"] = parse_accident_datetime(acc)
    print(f"Unique collision IDs: {len(acc)}")
    print(f"Parseable timestamps: {acc['dt'].notna().sum()}")
    print(f"Unparseable timestamps: {acc['dt'].isna().sum()}")
    if acc["dt"].isna().any():
        print("\nRows with unparseable timestamps:")
        print(acc.loc[acc["dt"].isna(), ["collision_id", "accidentdate", "time"]].to_string(index=False))

    print("\n=== SAME-COLLISION FEATURE REPLICATION ===")
    same_feature_rows = (
        enr.groupby("collision_id")[COUNT_COLS]
        .nunique(dropna=False)
        .max(axis=1)
        .eq(1)
        .mean()
    )
    print(f"Fraction of collision_ids where all 9 features are constant across duplicate rows: {same_feature_rows:.6f}")

    print("\n=== SAMPLE BRUTE-FORCE VERIFICATION ===")
    # Compare script output to exact counts for a small sample of accidents.
    valid = acc.dropna(subset=["dt"]).reset_index(drop=True)
    feat_unique = enr.drop_duplicates("collision_id").set_index("collision_id")[COUNT_COLS]

    all_dt = valid["dt"].to_numpy()
    all_x = valid["easting"].to_numpy(float)
    all_y = valid["northing"].to_numpy(float)
    radii = [50, 100, 200]
    years = [1, 2, 3]
    r_sq = {r: r * r for r in radii}

    def exact_counts(row):
        dt = pd.Timestamp(row["dt"])
        x = float(row["easting"])
        y = float(row["northing"])
        past = all_dt < np.datetime64(dt)
        dx = all_x - x
        dy = all_y - y
        d2 = dx * dx + dy * dy
        out = {}
        for yr in years:
            cutoff = np.datetime64((dt - pd.DateOffset(years=yr)).to_datetime64())
            within_year = past & (all_dt >= cutoff)
            for r in radii:
                out[f"past_accidents_{r}m_{yr}y"] = int(np.sum(within_year & (d2 <= r_sq[r])))
        return out

    sample_idx = np.linspace(0, len(valid) - 1, min(30, len(valid)), dtype=int)
    sample = valid.iloc[sample_idx].copy()
    mismatches = []
    for _, row in sample.iterrows():
        cid = row["collision_id"]
        expected = exact_counts(row)
        actual = feat_unique.loc[cid].to_dict()
        for c in COUNT_COLS:
            if expected[c] != actual[c]:
                mismatches.append((cid, c, expected[c], actual[c]))
                break

    if mismatches:
        print("MISMATCHES FOUND:")
        for m in mismatches[:10]:
            print(m)
    else:
        print("No mismatches found in the sample verification.")

    print("\n=== OUTPUT SUMMARY ===")
    print(enr[COUNT_COLS].describe().T[["min", "mean", "max"]].to_string())


if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("all_59k_accidents_complete_final_cleaned.csv", low_memory=False)

past_cols = [c for c in df.columns if c.startswith("past_accidents_")]

print("\n==============================")
print("PAST ACCIDENTS QA REPORT")
print("==============================\n")

# -----------------------------
# 1. BASIC STRUCTURE CHECK
# -----------------------------
print("1) COLUMN COUNT")
print("past_accidents columns:", len(past_cols))

# -----------------------------
# 2. COMPLETENESS CHECK
# -----------------------------
print("\n2) COMPLETENESS (NaNs)")
na_counts = df[past_cols].isna().sum().sum()
print("total NaNs:", na_counts)

rows_missing = df[past_cols].isna().any(axis=1).sum()
print("rows with any missing:", rows_missing)

# -----------------------------
# 3. ZERO/NEGATIVE CHECK
# -----------------------------
print("\n3) VALUE VALIDITY")
negatives = (df[past_cols] < 0).sum().sum()
print("negative values:", negatives)

zeros = (df[past_cols] == 0).sum().sum()
print("zero values:", zeros)

# -----------------------------
# 4. DUPLICATE COLLISION IDS
# -----------------------------
print("\n4) DUPLICATES")
dup = df["collision_id"].duplicated().sum()
print("duplicate collision_id rows:", dup)

# -----------------------------
# 5. FEATURE CONSISTENCY CHECK
# (severity breakdown <= total)
# -----------------------------
print("\n5) CONSISTENCY CHECK (severity <= total)")

issues = 0

for c in past_cols:
    if "_" not in c:
        continue

    parts = c.split("_")

    # skip non severity cols
    if len(parts) < 5:
        continue

    # extract matching total column
    base = "_".join(parts[:4])  # past_accidents_100m_1y
    sev = parts[4]              # slight/serious/fatal

    if sev not in ["slight", "serious", "fatal"]:
        continue

    total_col = base
    if total_col not in df.columns:
        continue

    if (df[c] > df[total_col]).sum() > 0:
        issues += 1

print("severity > total violations:", issues)

# -----------------------------
# 6. DISTRIBUTION SANITY CHECK
# -----------------------------
print("\n6) DISTRIBUTION STATS (sample)")
print(df[past_cols].describe().T[["mean", "min", "max"]].head(10))

# -----------------------------
# 7. SPARSITY CHECK
# -----------------------------
print("\n7) SPARSITY")
nonzero_ratio = (df[past_cols] > 0).sum().sum() / (df[past_cols].size)
print("non-zero ratio:", round(nonzero_ratio, 4))

# -----------------------------
# 8. FINAL PASS/FAIL
# -----------------------------
print("\n==============================")
if na_counts == 0 and negatives == 0 and issues == 0:
    print("✔ PASSED ALL QA CHECKS")
else:
    print("✖ ISSUES DETECTED")
print("==============================")

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("all_59k_accidents_complete_final.csv", low_memory=False)

print("\n====================")
print("CLEANING PIPELINE")
print("====================\n")

# -----------------------------
# 1. MISSING DATE/TIME
# -----------------------------
print("1) Missing accidentdate/time")
missing_dt = df["accidentdate"].isna() | df["time"].isna()
print("rows:", missing_dt.sum())

df = df[~missing_dt].copy()

# -----------------------------
# 2. FIX DATE / TIME CONSISTENCY
# -----------------------------
print("\n2) Recomputing canonical datetime")

df["accidentdate_parsed"] = pd.to_datetime(df["accidentdate"], errors="coerce", dayfirst=True)
df["time_str"] = df["time"].astype(str).str.strip()

df["dt"] = pd.to_datetime(
    df["accidentdate_parsed"].dt.strftime("%Y-%m-%d") + " " + df["time_str"],
    errors="coerce"
)

# -----------------------------
# 3. DAY OF WEEK FIX
# -----------------------------
print("\n3) Fixing day-of-week")
df["computed_day"] = df["dt"].dt.day_name()
mismatch = (df["day"].astype(str) != df["computed_day"].astype(str)).sum()
print("mismatches:", mismatch)

df["day"] = df["computed_day"]

# -----------------------------
# 4. WEATHER DATE ALIGNMENT CHECK
# -----------------------------
print("\n4) Weather alignment check")

# assume weather should align with date only
df["weather_date"] = df["dt"].dt.date

# flag mismatches if archive.daily.* exists
weather_cols = [c for c in df.columns if c.startswith("archive.daily.")]
if weather_cols:
    # we assume already joined, so just report consistency proxy
    print("archive.daily columns detected:", len(weather_cols))
else:
    print("no archive.daily columns found")

# -----------------------------
# 5. SPEEDLIMIT CLEANING
# -----------------------------
print("\n5) Speedlimit cleaning")

def parse_speed(x):
    if pd.isna(x):
        return np.nan
    x = str(x).lower().strip()

    if "20" in x:
        return 20
    if "30" in x:
        return 30
    if "40" in x:
        return 40
    if "50" in x:
        return 50
    if "60" in x:
        return 60
    if "70" in x:
        return 70
    try:
        return float(x)
    except:
        return np.nan

df["speedlimit_clean"] = df["speedlimit"].apply(parse_speed)
print("invalid speedlimit rows:", df["speedlimit_clean"].isna().sum())

# -----------------------------
# 6. IMD CLEANING
# -----------------------------
print("\n6) IMD validation")

imd_rank_invalid = ((df["IMDRank"] < 1) | (df["IMDRank"] > 32844)).sum()
imd_decile_invalid = ((df["IMDDecile"] < 1) | (df["IMDDecile"] > 10)).sum()

print("invalid IMDRank:", imd_rank_invalid)
print("invalid IMDDecile:", imd_decile_invalid)

df.loc[(df["IMDDecile"] < 1) | (df["IMDDecile"] > 10), "IMDDecile"] = np.nan
df.loc[(df["IMDRank"] < 1) | (df["IMDRank"] > 32844), "IMDRank"] = np.nan

# -----------------------------
# 7. LIGHT CONDITION CHECK
# -----------------------------
print("\n7) Light condition sanity check")

if "sunrise" in df.columns and "sunset" in df.columns:
    df["sunrise"] = pd.to_datetime(df["sunrise"], errors="coerce")
    df["sunset"] = pd.to_datetime(df["sunset"], errors="coerce")

    night_mismatch = (
        ((df["lightconditions_collapsed"] == "Dark") &
         (df["dt"].dt.time > df["sunrise"].dt.time) &
         (df["dt"].dt.time < df["sunset"].dt.time))
    ).sum()

    print("light condition mismatches:", night_mismatch)

# -----------------------------
# 8. FINAL SUMMARY
# -----------------------------
print("\n====================")
print("CLEANING COMPLETE")
print("====================")

print("\nRemaining rows:", len(df))
print("Columns:", df.shape[1])

# Save cleaned version
df.to_csv("all_59k_accidents_cleaned.csv", index=False)
print("\nSaved -> all_59k_accidents_cleaned.csv")

In [ ]:
import pandas as pd

df = pd.read_csv("all_59k_accidents_complete_final.csv", low_memory=False)

# isolate only past_accidents columns
cols = [c for c in df.columns if c.startswith("past_accidents_")]

print("Total past_accidents columns:", len(cols))

# 1) check missing values per column
na_counts = df[cols].isna().sum()

print("\n=== NA CHECK ===")
print(na_counts[na_counts > 0] if na_counts.sum() > 0 else "NO NaNs in past_accidents columns")

# 2) check if every row has values (strict check)
rows_with_any_na = df[cols].isna().any(axis=1).sum()

print("\nRows with ANY missing past_accidents value:", rows_with_any_na)

# 3) full completeness check
if rows_with_any_na == 0:
    print("\n✔ ALL ROWS HAVE COMPLETE past_accidents FEATURES")
else:
    print("\n✖ INCOMPLETE ROWS FOUND")

In [ ]:
import pandas as pd
from google.colab import files
from IPython.display import display

# Upload CSV
uploaded = files.upload()
file_name = next(iter(uploaded))

# Read CSV
df = pd.read_csv(file_name)

# Find columns whose names start with 'obj_'
obj_cols = [c for c in df.columns if str(c).startswith("obj_")]

# Build summary table
summary = []
for col in obj_cols:
    numeric_col = pd.to_numeric(df[col], errors="coerce").fillna(0)
    all_zero = (numeric_col == 0).all()

    summary.append({
        "column_name": col,
        "status": "removed" if all_zero else "kept",
        "reason": "all values are 0" if all_zero else "has non-zero values"
    })

summary_df = pd.DataFrame(summary)

# Show the table
display(summary_df)

# Save for easy copy into Google Docs / Sheets
summary_df.to_csv("obj_columns_summary.csv", index=False)

# Also print as text table in case you want to copy/paste
print(summary_df.to_string(index=False))

files.download("obj_columns_summary.csv")

In [ ]:
import pandas as pd

# Load your CSV
df = pd.read_csv("all_59k_accidents_archive_weather_poi_final_history.csv")
# Count unique collision_id values
unique_count = df["collision_id"].nunique()

print(f"Number of unique collision_id values: {unique_count}")

In [ ]:
import pandas as pd

# Files to compare
file1 = "all_59k_accidents_archive_weather_poi_final_history.csv"
file2 = "all_59k_accidents_complete.csv"

# Columns to check
cols_to_compare = [
    "past_accidents_50m_1y",
    "past_accidents_100m_3y",
]

# Load files
df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# Ensure same number of rows
if len(df1) != len(df2):
    print(f"Row count differs: {len(df1)} vs {len(df2)}")
    exit()

# Compare row-by-row
differences = []

for idx in range(len(df1)):
    for col in cols_to_compare:
        val1 = df1.loc[idx, col]
        val2 = df2.loc[idx, col]

        if pd.isna(val1) and pd.isna(val2):
            continue

        if val1 != val2:
            differences.append({
                "row": idx,
                "column": col,
                "file1_value": val1,
                "file2_value": val2,
            })

# Output results
if not differences:
    print("All values match.")
else:
    print(f"Found {len(differences)} differences:\n")

    for diff in differences:
        print(
            f"Row {diff['row']} | "
            f"{diff['column']} | "
            f"file1={diff['file1_value']} | "
            f"file2={diff['file2_value']}"
        )

In [ ]:
import pandas as pd

input_path = "all_59k_accidents_cleaned.csv"
output_path = "all_59k_accidents_no_history.csv"

df = pd.read_csv(input_path, low_memory=False)

# Find history columns
history_cols = [c for c in df.columns if c.startswith("past_accidents_")]

print("\nRemoving these columns:")
for c in history_cols:
    print(c)

print(f"\nTotal removed: {len(history_cols)}")

# Drop them
df = df.drop(columns=history_cols)

# Save
df.to_csv(output_path, index=False)

print(f"\nSaved cleaned file to: {output_path}")

In [ ]:
import csv

# Input and output file names
input_file = "distances_output.csv"
output_file = "output1_org.csv"

# Read the first row (headers) and first data row
with open(input_file, mode="r", newline="", encoding="utf-8") as infile:
    reader = csv.reader(infile)

    headers = next(reader)      # First row = headers
    first_row = next(reader)    # Second row = first data row
    second_row = next(reader)    # Second row = first data row

# Write headers + first row to new CSV
with open(output_file, mode="w", newline="", encoding="utf-8") as outfile:
    writer = csv.writer(outfile)

    writer.writerow(headers)
    writer.writerow(first_row)
    writer.writerow(second_row)

print(f"Created {output_file} with headers and first row.")

In [ ]:
import pandas as pd
import numpy as np

# Load CSV
df = pd.read_csv("all_59k_accidents_complete_final_cleaned.csv")

# Stats before replacement
nan_mask = df.isna()

nan_cells = nan_mask.sum().sum()
affected_rows = nan_mask.any(axis=1).sum()
affected_cols = nan_mask.any(axis=0).sum()

print(f"Total NaN cells: {nan_cells}")
print(f"Rows affected: {affected_rows}")
print(f"Columns affected: {affected_cols}")

# Show affected columns
print("\nAffected columns:")
print(df.columns[nan_mask.any(axis=0)].tolist())

# Print sample NaN locations (up to 20)
print("\nSample NaN values being replaced:")

sample_count = 0
max_samples = 20

for row_idx, row in df.iterrows():
    for col in df.columns:
        if pd.isna(row[col]):
            print(f"Row {row_idx}, Column '{col}' -> NaN")
            sample_count += 1

            if sample_count >= max_samples:
                break
    if sample_count >= max_samples:
        break

# Replace NaN with empty strings
df = df.replace({np.nan: ""})

# Save cleaned CSV
df.to_csv("output.csv", index=False)

print("\nNaN values replaced successfully.")

In [ ]:
import pandas as pd
from datetime import datetime, timezone

INPUT_CSV = "all_59k_accidents_complete_final_cleaned.csv"

def parse_source_dt(row):
    """Rebuild datetime from original columns"""
    raw_date = str(row["accidentdate"]).strip()
    raw_time = str(row["time"]).strip()

    date_part = datetime.strptime(raw_date, "%d-%b-%y")
    time_part = datetime.strptime(raw_time, "%H:%M")

    return datetime(
        date_part.year,
        date_part.month,
        date_part.day,
        time_part.hour,
        time_part.minute,
        tzinfo=timezone.utc,   # same assumption as your script
    )

def parse_output_dt(val):
    """Parse computed UTC column"""
    return datetime.fromisoformat(val.replace("Z", "+00:00"))

def main():
    df = pd.read_csv(INPUT_CSV, dtype=str)

    mismatches = []

    for idx, row in df.iterrows():
        try:
            src_dt = parse_source_dt(row)
            out_dt = parse_output_dt(row["accident_time_utc"])

            if src_dt != out_dt:
                mismatches.append({
                    "row": idx,
                    "accidentdate": row["accidentdate"],
                    "time": row["time"],
                    "expected_utc": src_dt.isoformat(),
                    "actual_utc": row["accident_time_utc"],
                })

        except Exception as e:
            mismatches.append({
                "row": idx,
                "error": str(e),
                "accidentdate": row.get("accidentdate"),
                "time": row.get("time"),
            })

    print(f"\nTotal mismatches: {len(mismatches)}\n")

    # print first 50 for inspection
    for m in mismatches[:50]:
        print(m)

    # optionally save
    pd.DataFrame(mismatches).to_csv("utc_mismatches.csv", index=False)

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd

FILE = "all_59k_accidents_complete_final_cleaned.csv"

df = pd.read_csv(FILE, low_memory=False)

# =========================================================
# normalize coordinates
# =========================================================

df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

# =========================================================
# count accidents per coordinate
# =========================================================

coord_counts = (
    df.groupby(["latitude", "longitude"])
      .size()
      .reset_index(name="count")
)

# keep only duplicated coordinates
dupes = coord_counts[coord_counts["count"] > 1]

print("\n================================================")
print("COORDINATES WITH MULTIPLE ACCIDENTS")
print("================================================")

print(f"Total duplicated coordinates: {len(dupes):,}")

# =========================================================
# inspect examples
# =========================================================

for i, row in dupes.head(20).iterrows():

    lat = row["latitude"]
    lon = row["longitude"]
    count = row["count"]

    print("\n================================================")
    print(f"LAT={lat}  LON={lon}")
    print(f"ACCIDENT COUNT: {count}")
    print("================================================")

    subset = df[
        (df["latitude"] == lat) &
        (df["longitude"] == lon)
    ].copy()

    cols = [
        "accidentdate",
        "time",
        "accident_time_utc"
    ]

    cols = [c for c in cols if c in subset.columns]

    print(subset[cols].to_string(index=False))

# =========================================================
# quantify corruption risk
# =========================================================

problematic = 0

for _, row in dupes.iterrows():

    lat = row["latitude"]
    lon = row["longitude"]

    subset = df[
        (df["latitude"] == lat) &
        (df["longitude"] == lon)
    ]

    source_times = (
        subset["accidentdate"].astype(str)
        + " "
        + subset["time"].astype(str)
    ).nunique()

    merged_times = subset["accident_time_utc"].nunique()

    # if many source accidents but only 1 merged weather time
    # that's evidence of merge corruption
    if source_times > 1 and merged_times == 1:
        problematic += 1

print("\n================================================")
print("MERGE CORRUPTION ANALYSIS")
print("================================================")

print(f"Duplicated coordinate groups: {len(dupes):,}")
print(f"Groups likely corrupted:     {problematic:,}")

In [ ]:
# Install dependency
!pip install -q pyproj

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pyproj import Geod
from google.colab import files

# Upload CSV


filename = "all_59k_accidents_complete_final_2_cleaned.csv"
df = pd.read_csv(filename)

# Verify required columns
required_columns = [
    "latitude",
    "longitude",
    "archive_retrieved_latitude",
    "archive_retrieved_longitude"
]

missing = [c for c in required_columns if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# Convert to numeric
for col in required_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Keep track of valid rows
valid_mask = df[required_columns].notna().all(axis=1)

# WGS84 geodesic calculator
geod = Geod(ellps="WGS84")

# Calculate distances (meters)
distances = np.full(len(df), np.nan)

_, _, valid_distances = geod.inv(
    df.loc[valid_mask, "longitude"].values,
    df.loc[valid_mask, "latitude"].values,
    df.loc[valid_mask, "archive_retrieved_longitude"].values,
    df.loc[valid_mask, "archive_retrieved_latitude"].values,
)

distances[valid_mask] = valid_distances

df["accident_to_weather_distance_m"] = distances

# Summary stats
print("=== Distance Statistics ===")
print(f"Rows analysed: {df['accident_to_weather_distance_m'].notna().sum():,}")
print(f"Average:       {df['accident_to_weather_distance_m'].mean():.2f} m")
print(f"Median:        {df['accident_to_weather_distance_m'].median():.2f} m")
print(f"Min:           {df['accident_to_weather_distance_m'].min():.2f} m")
print(f"Max:           {df['accident_to_weather_distance_m'].max():.2f} m")
print(f"Std Dev:       {df['accident_to_weather_distance_m'].std():.2f} m")

print("\n=== Percentiles ===")
for p in [50, 75, 90, 95, 99]:
    print(f"P{p}: {np.percentile(df['accident_to_weather_distance_m'].dropna(), p):.2f} m")

# Distribution table
bins = [0, 10, 25, 50, 100, 250, 500, 1000, 2500, 5000, np.inf]
labels = [
    "0-10m",
    "10-25m",
    "25-50m",
    "50-100m",
    "100-250m",
    "250-500m",
    "500m-1km",
    "1-2.5km",
    "2.5-5km",
    ">5km"
]

distribution = pd.cut(
    df["accident_to_weather_distance_m"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

dist_table = distribution.value_counts().sort_index().to_frame("count")
dist_table["percent"] = (
    dist_table["count"] / dist_table["count"].sum() * 100
).round(2)

print("\n=== Distribution ===")
print(dist_table)

# Histogram
plt.figure(figsize=(10, 5))
plt.hist(df["accident_to_weather_distance_m"].dropna(), bins=50)
plt.xlabel("Distance (meters)")
plt.ylabel("Count")
plt.title("Distance Distribution")
plt.show()

# Save output
output_file = "distances_output.csv"
df.to_csv(output_file, index=False)


In [ ]:
import pandas as pd
import numpy as np

def analyze_weather_missingness(file_path):
    print("Loading dataset...")
    # Read the full dataset
    df = pd.read_csv(file_path, low_memory=False)

    # 1. Parse or extract the hour from the time data
    # We prioritize 'dt' or 'time' if available
    if 'dt' in df.columns:
        df['accident_hour'] = pd.to_datetime(df['dt']).dt.hour
    elif 'time' in df.columns:
        # Gracefully handle string formats like '17:00' or '07:04'
        df['accident_hour'] = pd.to_datetime(df['time'], format='%H:%M', errors='coerce').dt.hour
    else:
        raise ValueError("Could not find a valid time or datetime column ('dt' or 'time') to extract hours.")

    # Filter rows where hour couldn't be parsed
    df = df.dropna(subset=['accident_hour'])
    df['accident_hour'] = df['accident_hour'].astype(int)

    # 2. Identify weather lag and rolling window columns
    # This captures both '_lag_1h', '_lag_3h', '_avg_last_3h', etc.
    lag_cols = [col for col in df.columns if 'lag' in col or 'avg_last' in col]

    if not lag_cols:
        print("No weather lag or rolling columns found in this dataset.")
        return

    print(f"Found {len(lag_cols)} weather lag/rolling columns to inspect.")

    # 3. Aggregate missingness (NaN count and percentage) grouped by Hour
    print("\n--- Summary of Missing Values (NaNs) by Hour ---")

    # Create a summary DataFrame
    summary_records = []

    for hour in sorted(df['accident_hour'].unique()):
        hour_df = df[df['accident_hour'] == hour]
        total_rows_at_hour = len(hour_df)

        # We can pick a couple of representative columns to demonstrate the pattern
        # e.g., temperature 1h lag and temperature 3h lag
        lag_1h_sample = [c for c in lag_cols if 'lag_1h' in c]
        lag_3h_sample = [c for c in lag_cols if 'lag_3h' in c]

        # Calculate NaN rates for these groups
        nan_rate_1h = hour_df[lag_1h_sample].isna().mean().mean() * 100 if lag_1h_sample else 0
        nan_rate_3h = hour_df[lag_3h_sample].isna().mean().mean() * 100 if lag_3h_sample else 0

        summary_records.append({
            'Hour': f"{hour:02d}:00",
            'Total Accidents': total_rows_at_hour,
            'Avg NaN % (1h Lags)': f"{nan_rate_1h:.1f}%",
            'Avg NaN % (3h Lags)': f"{nan_rate_3h:.1f}%"
        })

    summary_df = pd.DataFrame(summary_records)
    print(summary_df.to_string(index=False))

    # 4. Explicit verification of your hypothesis
    print("\n--- Hypothesis Validation Check ---")

    # Check Hour 00 for lag_1h
    h00_df = df[df['accident_hour'] == 0]
    lag_1h_cols = [c for c in lag_cols if 'lag_1h' in c]
    if len(h00_df) > 0 and lag_1h_cols:
        h00_nan_pct = h00_df[lag_1h_cols].isna().mean().mean() * 100
        print(f"• Hour 00:00 -> 1-hour lag columns are {h00_nan_pct:.1f}% missing.")
    else:
        print("• Hour 00:00 check skipped (no rows found or no lag_1h columns found).")

    # Check Hours 00, 01, 02 for lag_3h
    h00_02_df = df[df['accident_hour'].isin([0, 1, 2])]
    lag_3h_cols = [c for c in lag_cols if 'lag_3h' in c]
    if len(h00_02_df) > 0 and lag_3h_cols:
        h00_02_nan_pct = h00_02_df[lag_3h_cols].isna().mean().mean() * 100
        print(f"• Hours 00:00 - 02:00 -> 3-hour lag columns are {h00_02_nan_pct:.1f}% missing.")
    else:
        print("• Hours 00:00 - 02:00 check skipped (no rows found or no lag_3h columns found).")

if __name__ == "__main__":
    # Replace 'your_full_dataset.csv' with the actual path to your full file
    file_name = 'all_59k_accidents_complete_final_2_cleaned.csv'
    analyze_weather_missingness(file_name)

In [ ]:
import pandas as pd

def verify_timezone_hypothesis(file_path):
    print("Loading dataset...")
    df = pd.read_csv(file_path, low_memory=False)

    # 1. Parse datetime to get both Hour and Month
    if 'dt' in df.columns:
        df['datetime_parsed'] = pd.to_datetime(df['dt'])
    elif 'accidentdate' in df.columns:
        # Fallback to accidentdate if dt isn't standard
        df['datetime_parsed'] = pd.to_datetime(df['accidentdate'])
    else:
        raise ValueError("Could not find a datetime column to parse.")

    df['accident_hour'] = df['datetime_parsed'].dt.hour
    df['accident_month'] = df['datetime_parsed'].dt.month

    # 2. Get the 1-hour lag columns
    lag_1h_cols = [c for c in df.columns if 'lag_1h' in c]
    if not lag_1h_cols:
        print("No 'lag_1h' columns found.")
        return

    # 3. Filter specifically for midnight accidents (Hour 00)
    midnight_df = df[df['accident_hour'] == 0]

    if len(midnight_df) == 0:
        print("No accidents found at Hour 00:00 in this dataset.")
        return

    print("\n--- Breakdown of Midnight (00:00) Missingness by Month ---")

    # Group by month and calculate the average missing percentage
    monthly_summary = []
    month_names = {
        1: "January (GMT)", 2: "February (GMT)", 3: "March (GMT/BST)",
        4: "April (BST)", 5: "May (BST)", 6: "June (BST)",
        7: "July (BST)", 8: "August (BST)", 9: "September (BST)",
        10: "October (BST/GMT)", 11: "November (GMT)", 12: "December (GMT)"
    }

    for month in range(1, 13):
        month_df = midnight_df[midnight_df['accident_month'] == month]
        total_accidents = len(month_df)

        if total_accidents > 0:
            nan_pct = month_df[lag_1h_cols].isna().mean().mean() * 100
        else:
            nan_pct = float('nan')

        monthly_summary.append({
            'Month': month_names[month],
            'Midnight Accidents': total_accidents,
            'Lag_1h Missing %': f"{nan_pct:.1f}%" if not pd.isna(nan_pct) else "No Data"
        })

    print(pd.DataFrame(monthly_summary).to_string(index=False))

if __name__ == "__main__":
    # Change this to your full dataset file path
    verify_timezone_hypothesis('all_59k_accidents_complete_final_2_cleaned.csv')

In [ ]:
import pandas as pd

def extract_early_morning_gaps(file_path):
    print("Loading dataset...")
    df = pd.read_csv(file_path, low_memory=False)

    # Parse datetime and extract hour
    df['datetime_parsed'] = pd.to_datetime(df['dt'], errors='coerce')
    df['accident_hour'] = df['datetime_parsed'].dt.hour

    # Filter for Early Morning (00:00 to 05:59)
    early_df = df[df['accident_hour'].isin([0, 1, 2, 3, 4, 5])].copy()

    # Required lag columns
    lag_1h_col = 'archive.hourly.temperature_2m_lag_1h'
    lag_3h_col = 'archive.hourly.temperature_2m_lag_3h'
    lag_6h_col = 'archive.hourly.temperature_2m_lag_6h'

    # Validate columns exist
    required_cols = [lag_1h_col, lag_3h_col, lag_6h_col]
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        print("\nERROR: Missing required columns:")
        for col in missing_cols:
            print(f"  - {col}")

        print("\nAvailable temperature_2m columns:")
        for col in sorted([c for c in df.columns if 'temperature_2m' in c]):
            print(f"  - {col}")

        raise KeyError("Required lag columns not found.")

    print(f"Using 1h lag column: {lag_1h_col}")
    print(f"Using 3h lag column: {lag_3h_col}")
    print(f"Using 6h lag column: {lag_6h_col}")

    print("\n=========================================================================================")
    print(
        f"{'Hour':<6} | {'Total':<6} | "
        f"{'Missing 1h Lag':<15} | "
        f"{'Missing 3h Lag':<15} | "
        f"{'Missing 6h Lag':<15}"
    )
    print("-" * 89)

    for hr in range(6):
        hr_df = early_df[early_df['accident_hour'] == hr]
        total = len(hr_df)

        if total == 0:
            print(
                f"{hr:02d}:00  | 0      | "
                f"0 (0.0%)        | "
                f"0 (0.0%)        | "
                f"0 (0.0%)"
            )
            continue

        m_1h = hr_df[lag_1h_col].isna().sum()
        m_3h = hr_df[lag_3h_col].isna().sum()
        m_6h = hr_df[lag_6h_col].isna().sum()

        pct_1h = (m_1h / total) * 100
        pct_3h = (m_3h / total) * 100
        pct_6h = (m_6h / total) * 100

        print(
            f"{hr:02d}:00  | {total:<6} | "
            f"{m_1h:<4} ({pct_1h:.1f}%)     | "
            f"{m_3h:<4} ({pct_3h:.1f}%)     | "
            f"{m_6h:<4} ({pct_6h:.1f}%)"
        )

    print("=========================================================================================\n")

    # Rows where any lag is missing
    condition = (
        early_df[lag_1h_col].isna()
        | early_df[lag_3h_col].isna()
        | early_df[lag_6h_col].isna()
    )

    failing_rows = early_df[condition]

    coord_cols = [
        'accidentdate',
        'time',
        'dt',
        'latitude',
        'longitude'
    ]
    coord_cols = [c for c in coord_cols if c in df.columns]

    print(f"Found {len(failing_rows)} rows facing lag-boundary issues.")

    output_file = 'failing_weather_coordinates.csv'
    print(f"Exporting these rows to '{output_file}'...")

    failing_rows[coord_cols].to_csv(output_file, index=False)

    print("\n--- Top 10 Sample Rows with Coordinates ---")
    print(failing_rows[coord_cols].head(10).to_string(index=False))

    # Extra diagnostic: rows where 3h is missing but 6h is not
    weird_cases = early_df[
        early_df[lag_3h_col].isna()
        & early_df[lag_6h_col].notna()
    ]

    print(
        f"\nRows where 3h lag is missing but 6h lag exists: "
        f"{len(weird_cases)}"
    )

    if len(weird_cases) > 0:
        print(
            weird_cases[
                ['dt', lag_1h_col, lag_3h_col, lag_6h_col]
            ].head(10).to_string(index=False)
        )


if __name__ == "__main__":
    extract_early_morning_gaps(
        'all_59k_accidents_complete_final_2_cleaned.csv'
    )

In [ ]:
import pandas as pd

def extract_early_morning_rows(file_path):
    print("Loading dataset...")
    df = pd.read_csv(file_path, low_memory=False)

    # Parse datetime
    df['datetime_parsed'] = pd.to_datetime(df['dt'], errors='coerce')
    df['accident_hour'] = df['datetime_parsed'].dt.hour

    # Filter rows from 00:00 to 05:59
    early_df = df[df['accident_hour'] < 6].copy()

    print("\n============================================================")
    print("Early Morning Accidents (00:00 - 05:59)")
    print("============================================================")
    print(f"Total rows: {len(early_df)}\n")

    print(f"{'Hour':<6} | {'Rows':<10}")
    print("-" * 20)

    for hr in range(6):
        count = (early_df['accident_hour'] == hr).sum()
        print(f"{hr:02d}:00  | {count}")

    print("============================================================\n")

    # Columns to export
    export_cols = [
        'accidentdate',
        'time',
        'dt',
        'latitude',
        'longitude'
    ]

    # Keep only columns that actually exist
    export_cols = [c for c in export_cols if c in df.columns]

    output_file = 'all_rows_00_to_06.csv'

    print(f"Exporting {len(early_df)} rows to '{output_file}'...")

    early_df[export_cols].to_csv(output_file, index=False)

    print("\n--- Top 10 Sample Rows ---")
    print(early_df[export_cols].head(10).to_string(index=False))

    print(f"\nDone. Saved {len(early_df)} rows.")

if __name__ == "__main__":
    extract_early_morning_rows(
        "all_59k_accidents_complete_final_2_cleaned.csv"
    )

In [ ]:
#!/usr/bin/env python3
# Extract weather and matching accident from jsonl

import json
import pandas as pd
from datetime import datetime
import numpy as np
# ============================================================
# CONFIG
# ============================================================

INPUT_JSONL = "open_meteo_archive_only (1).jsonl"
OUTPUT_CSV = "open_meteo_archive_final_utc.csv"

def circular_mean(sin_vals, cos_vals):
    if len(sin_vals) == 0:
        return None, None

    sin_mean = sum(sin_vals) / len(sin_vals)
    cos_mean = sum(cos_vals) / len(cos_vals)

    norm = (sin_mean**2 + cos_mean**2) ** 0.5
    if norm == 0:
        return None, None

    sin_mean /= norm
    cos_mean /= norm

    return sin_mean, cos_mean

def safe_get(v, i):
    if i is None:
        return None
    if not isinstance(v, list):
        return None
    if i < 0 or i >= len(v):
        return None
    return v[i]


def clean(vals):
    out = []
    for x in vals:
        try:
            if x is None:
                continue
            out.append(float(x))
        except:
            continue
    return out


def mean(values):
    vals = clean(values)
    return sum(vals) / len(vals) if vals else None

def to_rad(x):
    if x is None:
        return None
    try:
        return np.deg2rad(float(x) % 360)
    except:
        return None


def summ(values):
    vals = clean(values)
    return sum(vals) if vals else None
# ============================================================
# MAIN
# ============================================================

rows = []

with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        obj = json.loads(line)

        row = {}

        # ====================================================
        # TOP-LEVEL FIELDS
        # ====================================================

        for k, v in obj.items():
            if k == "archive_json":
                continue

            row[k] = v

        # ====================================================
        # ARCHIVE SECTION
        # ====================================================

        archive = obj.get("archive_json", {})

        for k, v in archive.items():

            # Skip nested hourly for separate handling
            if k == "hourly":
                continue

            # Skip hourly_units metadata
            if k == "hourly_units":
                continue

            row[f"archive_{k}"] = v

        # ====================================================
        # HOURLY WEATHER VARIABLES
        # ====================================================

        hourly = archive.get("hourly", {})

        times = hourly.get("time", [])

        target = datetime.fromisoformat(
            obj["accident_time_utc"].replace("Z", "+00:00")
        )

        match_idx = None

        for i, t in enumerate(times):
            dt = datetime.fromisoformat(t + ":00+00:00")
            if dt <= target:
                match_idx = i
            else:
                break

        if match_idx is None:
            continue

        for k, v in hourly.items():

            if k == "time":
                if match_idx is not None:
                    row["weather_time"] = times[match_idx]
                continue

            if not isinstance(v, list):
                continue

            # -----------------------------
            # base value (matched hour)
            # -----------------------------
            base_val = safe_get(v, match_idx)
            row[k] = base_val
            # -----------------------------
            # wind direction sin/cos (base + rolling)
            # -----------------------------
            if k in ["winddirection_10m", "winddirection_100m"]:

                # ---- base ----
                r = to_rad(base_val)
                row[f"{k}_sin"] = np.sin(r) if r is not None else None
                row[f"{k}_cos"] = np.cos(r) if r is not None else None

                # ---- rolling windows (correctly using time-aligned values) ----
                sin_3, cos_3 = [], []
                sin_6, cos_6 = [], []

                for j in range(3):
                    r = to_rad(safe_get(v, match_idx - j))
                    if r is not None:
                        sin_3.append(np.sin(r))
                        cos_3.append(np.cos(r))

                for j in range(6):
                    r = to_rad(safe_get(v, match_idx - j))
                    if r is not None:
                        sin_6.append(np.sin(r))
                        cos_6.append(np.cos(r))

                sin_m_3, cos_m_3 = circular_mean(sin_3, cos_3)
                row[f"{k}_sin_avg_last_3h"] = sin_m_3
                row[f"{k}_cos_avg_last_3h"] = cos_m_3
                row[f"{k}_dir_avg_last_3h"] = (np.degrees(np.arctan2(sin_m_3, cos_m_3)) + 360) % 360 if sin_m_3 is not None else None
                sin_m_6, cos_m_6 = circular_mean(sin_6, cos_6)
                row[f"{k}_sin_avg_last_6h"] = sin_m_6
                row[f"{k}_cos_avg_last_6h"] = cos_m_6
                row[f"{k}_dir_avg_last_6h"] = (np.degrees(np.arctan2(sin_m_6, cos_m_6)) + 360) % 360 if sin_m_6 is not None else None

            # -----------------------------
            # lag features
            # -----------------------------
            row[f"{k}_lag_1h"] = safe_get(v, match_idx - 1)
            row[f"{k}_lag_3h"] = safe_get(v, match_idx - 3)

            # -----------------------------
            # rolling windows
            # -----------------------------
            window_3h = [
                safe_get(v, match_idx - i)
                for i in range(3)
            ]

            window_6h = [
                safe_get(v, match_idx - i)
                for i in range(6)
            ]

            # avg vs sum logic (simple rule-based)
            is_avg = k in [
                "temperature_2m",
                "relativehumidity_2m",
                "dewpoint_2m",
                "apparent_temperature",
                "weathercode",
                "pressure_msl",
                "surface_pressure",
                "cloudcover",
                "cloudcover_low",
                "cloudcover_mid",
                "cloudcover_high",
                "windspeed_10m",
                "winddirection_10m",
                "windgusts_10m",
                "et0_fao_evapotranspiration",
                "vapour_pressure_deficit",
                "windspeed_100m",
                "winddirection_100m",
                "soil_temperature_0_to_7cm",
                "soil_temperature_7_to_28cm",
                "soil_temperature_28_to_100cm",
                "soil_temperature_100_to_255cm",
                "soil_moisture_0_to_7cm",
                "soil_moisture_7_to_28cm",
                "soil_moisture_28_to_100cm",
                "soil_moisture_100_to_255cm"
            ]

            if is_avg:
                row[f"{k}_avg_last_3h"] = mean(window_3h)
                row[f"{k}_avg_last_6h"] = mean(window_6h)
                row[f"{k}_sum_last_3h"] = summ(window_3h)
                row[f"{k}_sum_last_6h"] = summ(window_6h)
            else:
                row[f"{k}_sum_last_3h"] = summ(window_3h)
                row[f"{k}_sum_last_6h"] = summ(window_6h)

        rows.append(row)

# ============================================================
# EXPORT
# ============================================================

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved {len(df):,} rows")
print(f"Saved CSV -> {OUTPUT_CSV}")
print(f"Columns -> {len(df.columns)}")

print("\nColumns:")
for c in df.columns:
    print("-", c)

In [ ]:
import logging
import pandas as pd

# =========================
# FILE PATHS
# =========================
file1_path = "all_59k_accidents_complete_final_2_cleaned.csv"
file2_path = "open_meteo_archive_missing_clean.csv"
output_path = "all_59k_accidents_complete_final_2_cleaned_updated.csv"

FILE1_LAT = "latitude"
FILE1_LON = "longitude"
FILE1_DT  = "dt"

FILE2_LAT = "source_latitude"
FILE2_LON = "source_longitude"
FILE2_DT  = "accident_datetime_utc"

# =========================
# JOIN KEYS
# =========================
KEY_LAT = "_k_lat"
KEY_LON = "_k_lon"
KEY_DT  = "_k_dt"
KEY_COLS = [KEY_LAT, KEY_LON, KEY_DT]

# =========================
# LOGGING
# =========================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger("merge")

# =========================
# LOAD FILES
# =========================
logger.info("Loading files...")

df1 = pd.read_csv(file1_path, low_memory=False)
df2 = pd.read_csv(file2_path, low_memory=False)

logger.info(f"file1 rows={len(df1):,}")
logger.info(f"file2 rows={len(df2):,}")

# =========================
# NORMALIZATION
# =========================
def norm_latlon(s):
    return (
        pd.to_numeric(s, errors="coerce")
        .map(lambda x: format(x, ".10f") if pd.notnull(x) else None)
    )

def norm_dt(s):
    return (
        pd.to_datetime(s, errors="coerce", utc=True)
        .dt.strftime("%Y-%m-%dT%H:%M:%S+00:00")
    )

df1[KEY_LAT] = norm_latlon(df1[FILE1_LAT])
df1[KEY_LON] = norm_latlon(df1[FILE1_LON])
df1[KEY_DT]  = norm_dt(df1[FILE1_DT])

df2[KEY_LAT] = norm_latlon(df2[FILE2_LAT])
df2[KEY_LON] = norm_latlon(df2[FILE2_LON])
df2[KEY_DT]  = norm_dt(df2[FILE2_DT])

# =========================
# SHARED COLUMNS
# =========================
shared_cols = sorted(
    (set(df1.columns) & set(df2.columns))
    - set(KEY_COLS)
)

logger.info(f"Shared columns found: {len(shared_cols)}")

# =========================
# DEDUPE FILE2 BY KEY
# =========================
# Keeps one record per normalized key.
# Then every matching row in file1 gets those values.
df2_lookup = (
    df2[KEY_COLS + shared_cols]
    .drop_duplicates(subset=KEY_COLS)
)

logger.info(
    f"file2 unique key rows={len(df2_lookup):,}"
)

# =========================
# MERGE
# =========================
logger.info("Merging...")

merged = df1.merge(
    df2_lookup,
    on=KEY_COLS,
    how="left",
    indicator=True,
    validate="many_to_one",
    suffixes=("", "_src")
)

matched = merged["_merge"].eq("both")

matched_rows = int(matched.sum())
unmatched_rows = int((~matched).sum())

logger.info(f"Matched rows: {matched_rows:,}")
logger.info(f"Unmatched rows: {unmatched_rows:,}")

# =========================
# COPY VALUES
# =========================
total_replacements = 0
columns_changed = []

for col in shared_cols:

    src_col = f"{col}_src"

    # Only replace when file2 actually has a value
    replace_mask = matched & merged[src_col].notna()

    before = merged.loc[replace_mask, col]
    after  = merged.loc[replace_mask, src_col]

    changed_mask = ~(
        before.eq(after)
        | (before.isna() & after.isna())
    )

    replacements = int(changed_mask.sum())

    merged.loc[replace_mask, col] = after

    total_replacements += replacements
    columns_changed.append((col, replacements))

    logger.info(
        f"Column '{col}': {replacements:,} replacements"
    )

logger.info(
    f"Total replacements made: {total_replacements:,}"
)

# =========================
# CLEANUP
# =========================
helper_cols = (
    KEY_COLS
    + ["_merge"]
    + [f"{c}_src" for c in shared_cols]
)

updated_df = merged.drop(
    columns=[c for c in helper_cols if c in merged.columns]
)

# =========================
# SAVE
# =========================
updated_df.to_csv(output_path, index=False)

logger.info(f"Saved: {output_path}")

# =========================
# SUMMARY
# =========================
print("\n==============================")
print("SYNC SUMMARY")
print("==============================")

print(f"Rows in file1:      {len(df1):,}")
print(f"Rows in file2:      {len(df2):,}")
print(f"Matched rows:       {matched_rows:,}")
print(f"Unmatched rows:     {unmatched_rows:,}")
print(f"Total replacements: {total_replacements:,}")

print("\nPER-COLUMN REPLACEMENTS")
print("-----------------------")

for col, n in sorted(
    columns_changed,
    key=lambda x: x[1],
    reverse=True
):
    print(f"{col}: {n:,}")

In [ ]:
import pandas as pd

input_file = "all_59k_accidents_complete_final_2_cleaned_updated.csv"
output_file = "subset.csv"

df = pd.read_csv(input_file)

# Convert values to numbers where possible
rain = pd.to_numeric(df["archive.hourly.rain"], errors="coerce")

# Keep rows where rain equals zero
subset = df[rain == 0]

subset.to_csv(output_file, index=False)

print(f"Saved {len(subset)} rows to {output_file}")

In [ ]:
import pandas as pd

input_file = "subset.csv"
output_file = "unique_rows.csv"

# Load the CSV
df = pd.read_csv(input_file)

# Keep only the first occurrence of each unique
# (latitude, longitude, dt) combination
unique_df = df.drop_duplicates(
    subset=["latitude", "longitude", "dt"],
    keep="first"
)

# Save the result
unique_df.to_csv(output_file, index=False)

print(
    f"Input rows: {len(df)}\n"
    f"Unique rows: {len(unique_df)}\n"
    f"Saved to: {output_file}"
)

In [ ]:
import pandas as pd

csv_file = "all_59k_accidents_complete_final_2_cleaned.csv"  # Replace with your CSV path

# Read CSV
df = pd.read_csv(csv_file)

# Columns to remove
cols_to_drop = [
    "vehicletypebanded",
    "computed_day",
    "time_str"
]

# Drop only columns that exist
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Save back to the same file
df.to_csv(csv_file, index=False)

print(f"Updated CSV saved: {csv_file}")

In [ ]:
import pandas as pd

# Load CSV
df = pd.read_csv("/content/all_59k_accidents_complete_final_2_cleaned.csv")

# Parse the datetime column
df["dt"] = pd.to_datetime(df["dt"], errors="coerce")

def convert_london_to_utc(ts):
    if pd.isna(ts):
        return pd.NaT

    try:
        return (
            ts.tz_localize(
                "Europe/London",
                ambiguous="raise",
                nonexistent="shift_forward"
            )
            .tz_convert("UTC")
            .tz_localize(None)
        )
    except Exception:
        # Ambiguous timestamp -> leave unchanged
        return ts

df["dt_utc"] = df["dt"].apply(convert_london_to_utc)

df.to_csv(
    "/content/all_59k_accidents_complete_final_2_cleaned_utc.csv",
    index=False
)

print("Done")

In [ ]:
import pandas as pd

# Load CSV
df = pd.read_csv("/content/all_59k_accidents_complete_final_2_cleaned_utc_no_accident_time.csv")

# Rename column
df.rename(columns={"dt_utc": "accident_time_utc"}, inplace=True)

# Save result
df.to_csv(
    "/content/all_59k_accidents_complete_final_2_cleaned_utc_final.csv",
    index=False
)

print("Done - column renamed to accident_time_utc")

In [ ]:
# Convert jsonl to csv
#!/usr/bin/env python3

import json
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

INPUT_JSONL = "open_meteo_archive_precipitation_backtracked.jsonl"
OUTPUT_CSV = "open_meteo_archive_precipitation_backtracked.csv"


# ============================================================
# HELPERS
# ============================================================

def first_or_none(value):
    if isinstance(value, list):
        return value[0] if value else None
    return value


# ============================================================
# MAIN
# ============================================================

rows = []

with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        obj = json.loads(line)

        row = {}

        # ====================================================
        # TOP-LEVEL FIELDS
        # ====================================================

        for k, v in obj.items():
            if k == "archive":
                continue

            row[k] = v

        # ====================================================
        # ARCHIVE SECTION
        # ====================================================

        archive = obj.get("archive", {})

        for k, v in archive.items():

            # Skip nested hourly for separate handling
            if k == "hourly":
                continue

            # Skip hourly_units metadata
            if k == "hourly_units":
                continue

            row[f"archive_{k}"] = v

        # ====================================================
        # HOURLY WEATHER VARIABLES
        # ====================================================

        hourly = archive.get("hourly", {})

        for k, v in hourly.items():
            row[k] = first_or_none(v)

        rows.append(row)

# ============================================================
# EXPORT
# ============================================================

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved {len(df):,} rows")
print(f"Saved CSV -> {OUTPUT_CSV}")
print(f"Columns -> {len(df.columns)}")

print("\nColumns:")
for c in df.columns:
    print("-", c)

In [ ]:
import pandas as pd
import ast

INPUT_CSV = "open_meteo_archive_precipitation_backtracked.csv"
OUTPUT_CSV = "weather_extracted_simple.csv"

df = pd.read_csv(INPUT_CSV)

rows = []

for _, row in df.iterrows():

    if str(row.get("archive_status")) != "ok":
        continue

    try:
        weather = ast.literal_eval(row["archive_selected_weather"])

        if not isinstance(weather, dict):
            continue

        rows.append({
            "_row_index": row["_row_index"],
            "accident_datetime_utc": row["accident_datetime_utc"],
            "source_latitude": row["source_latitude"],
            "source_longitude": row["source_longitude"],

            "hours_back": weather.get("hours_back"),
            "precipitation": weather.get("precipitation"),
            "rain": weather.get("rain"),
            "snowfall": weather.get("snowfall"),
        })

    except Exception:
        continue


out = pd.DataFrame(rows)
out.to_csv(OUTPUT_CSV, index=False)

print(f"Saved {len(out):,} rows -> {OUTPUT_CSV}")

In [ ]:
import pandas as pd

# Load CSV
df = pd.read_csv("/content/weather_extracted_simple.csv")

# Parse the datetime column
df["accident_datetime_utc"] = pd.to_datetime(df["accident_datetime_utc"], errors="coerce")

def convert_london_to_utc(ts):
    if pd.isna(ts):
        return pd.NaT

    try:
        return (
            ts.tz_localize(
                "Europe/London",
                ambiguous="raise",
                nonexistent="shift_forward"
            )
            .tz_convert("UTC")
            .tz_localize(None)
        )
    except Exception:
        # Ambiguous timestamp -> leave unchanged
        return ts

df["dt_utc"] = df["accident_datetime_utc"].apply(convert_london_to_utc)

df.to_csv(
    "/content/weather_extracted_simple_utc.csv",
    index=False
)

print("Done")

In [ ]:
import logging
import pandas as pd

# =========================
# FILE PATHS
# =========================
file1_path = "all_59k_accidents_complete_final_2_cleaned.csv"
file2_path = "corrected_last_precipitation.csv"
output_path = "all_59k_accidents_complete_final_2_cleaned_updated.csv"

# =========================
# KEYS
# =========================
FILE1_LAT = "latitude"
FILE1_LON = "longitude"
FILE1_DT  = "accident_time_utc"

FILE2_LAT = "source_latitude"
FILE2_LON = "source_longitude"
FILE2_DT  = "accident_datetime_utc"

KEY_LAT = "_k_lat"
KEY_LON = "_k_lon"
KEY_DT  = "_k_dt"
KEY_COLS = [KEY_LAT, KEY_LON, KEY_DT]

# =========================
# LOGGING
# =========================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger("merge")

# =========================
# LOAD DATA
# =========================
logger.info("Loading files...")

df1 = pd.read_csv(file1_path, low_memory=False)
df2 = pd.read_csv(file2_path, low_memory=False)

logger.info(f"file1 rows: {len(df1):,}")
logger.info(f"file2 rows: {len(df2):,}")

# =========================
# FIX FILE2 TIME FORMAT (MATCH FILE1 EXACTLY)
# =========================
df2[FILE2_DT] = (
    pd.to_datetime(df2[FILE2_DT], errors="coerce", utc=True)
      .dt.strftime("%Y-%m-%d %H:%M:%S")
)

# =========================
# BUILD STRING KEYS
# =========================
df1[KEY_LAT] = df1[FILE1_LAT].astype(str)
df1[KEY_LON] = df1[FILE1_LON].astype(str)
df1[KEY_DT]  = df1[FILE1_DT].astype(str)

df2[KEY_LAT] = df2[FILE2_LAT].astype(str)
df2[KEY_LON] = df2[FILE2_LON].astype(str)
df2[KEY_DT]  = df2[FILE2_DT].astype(str)

# =========================
# DEBUG: ROW_INDEX → FILE1 ROW COMPARISON
# =========================
logger.info("Checking _row_index alignment...")

assert "_row_index" in df2.columns, "Missing _row_index in file2"

bad_matches = []

for i, r in df2.iterrows():
    idx = r["_row_index"]

    if pd.isna(idx) or int(idx) >= len(df1):
        continue

    f1 = df1.iloc[int(idx)]

    f1_key = (str(f1[FILE1_LAT]), str(f1[FILE1_LON]), str(f1[FILE1_DT]))
    f2_key = (r[KEY_LAT], r[KEY_LON], r[KEY_DT])

    if f1_key != f2_key:
        bad_matches.append((idx, f1_key, f2_key))

# =========================
# PRINT DEBUG RESULTS
# =========================
print("\n==============================")
print("_ROW_INDEX MISMATCH CHECK")
print("==============================")

print(f"Total mismatches: {len(bad_matches):,}")

print("\n--- Sample mismatches (3–4) ---")
for idx, f1_key, f2_key in bad_matches[:4]:
    print("\nROW_INDEX:", idx)
    print("FILE1:", f1_key)
    print("FILE2:", f2_key)

# =========================
# BUILD LOOKUP (NORMAL MERGE)
# =========================
df2_lookup = (
    df2[
        KEY_COLS + ["hours_back", "precipitation"]
    ]
    .rename(columns={
        "hours_back": "hours_back_precipitation",
        "precipitation": "back_precipitation",
    })
    .drop_duplicates(subset=KEY_COLS)
)

# =========================
# MERGE
# =========================
logger.info("Merging...")

merged = df1.merge(
    df2_lookup,
    on=KEY_COLS,
    how="left",
    validate="many_to_one"
)

# =========================
# STATS
# =========================
matched = merged["hours_back_precipitation"].notna()

logger.info(f"Matched rows: {matched.sum():,}")
logger.info(f"Unmatched rows: {(~matched).sum():,}")

# =========================
# SAVE
# =========================
merged.drop(columns=KEY_COLS, inplace=True)
merged.to_csv(output_path, index=False)

logger.info(f"Saved -> {output_path}")

# =========================
# SUMMARY
# =========================
print("\n==============================")
print("DONE")
print("==============================")
print(f"Rows file1: {len(df1):,}")
print(f"Rows file2: {len(df2):,}")
print(f"Matched:    {matched.sum():,}")
print(f"Unmatched:  {(~matched).sum():,}")

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

# Upload file
csv_file = "all_59k_accidents_with_past_precip.csv"

# Read data
df = pd.read_csv(csv_file, low_memory=False)

# Clean columns
df["easting"] = pd.to_numeric(df["easting"], errors="coerce")
df["northing"] = pd.to_numeric(df["northing"], errors="coerce")
df["year"] = pd.to_numeric(df["year"], errors="coerce")

df = df.dropna(subset=["easting", "northing", "year", "casualty_severity"]).reset_index(drop=True)

# Severity score: Serious/Fatal = 1, others = 0
df["severity_score"] = df["casualty_severity"].astype(str).str.strip().str.lower().isin(
    ["serious", "fatal"]
).astype(int)

# Output array
feature = np.zeros(len(df), dtype=float)

# Group row indices by year
year_groups = df.groupby("year").indices
years_sorted = sorted(year_groups.keys())

# Batch size to keep memory safe
batch_size = 1000

for year in years_sorted:
    prev_year = year - 1
    if prev_year not in year_groups:
        continue

    curr_idx = np.array(year_groups[year])
    prev_idx = np.array(year_groups[prev_year])

    curr_coords = df.loc[curr_idx, ["easting", "northing"]].to_numpy(dtype=float)
    prev_coords = df.loc[prev_idx, ["easting", "northing"]].to_numpy(dtype=float)
    prev_severity = df.loc[prev_idx, "severity_score"].to_numpy(dtype=float)

    # Process current year rows in chunks
    for start in range(0, len(curr_idx), batch_size):
        end = min(start + batch_size, len(curr_idx))
        chunk_coords = curr_coords[start:end]

        # Pairwise distances: shape = (chunk_size, prev_count)
        dx = chunk_coords[:, 0][:, None] - prev_coords[:, 0][None, :]
        dy = chunk_coords[:, 1][:, None] - prev_coords[:, 1][None, :]
        dist = np.sqrt(dx * dx + dy * dy)

        # Avoid divide-by-zero
        dist[dist == 0] = np.nan

        # Since we only look at the previous year, years_ago = 1
        contrib = prev_severity[None, :] / np.sqrt(dist)

        feature[curr_idx[start:end]] = np.nansum(contrib, axis=1)

df["weighted_prior_severe_memory"] = feature

# Save result
out_file = "output_with_feature.csv"
df.to_csv(out_file, index=False)

In [ ]:
import pandas as pd

# Read CSV
df = pd.read_csv("output_with_feature.csv", low_memory=False)

# Drop the column if it exists
df = df.drop(columns=["severity_score"], errors="ignore")

# Save back out
df.to_csv("output_with_feature_no_severity_score.csv", index=False)

print("Done. Saved to output_with_feature_no_severity_score.csv")

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

# ----------------------------
# Upload and read CSV
# ----------------------------

csv_file = "all_59k_accidents_with_past_precip_weighted_severity.csv"

df = pd.read_csv(csv_file, low_memory=False)

# ----------------------------
# Window definitions
# ----------------------------
years = [1, 2, 3, 4, 5]
radii = [50, 100, 200, 300, 400, 500]

def col_name(y, r):
    return f"past_accidents_{r}m_{y}y"

# ----------------------------
# Make sure all needed columns are numeric
# ----------------------------
for y in years:
    for r in radii:
        c = col_name(y, r)
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

# ----------------------------
# Create hotspot gradient features
# (1 + local) / (1 + broad)
# and inverse version too
# ----------------------------
created_cols = []

for ly in years:
    for lr in radii:
        local_col = col_name(ly, lr)
        if local_col not in df.columns:
            continue

        for by in years:
            for br in radii:
                broad_col = col_name(by, br)
                if broad_col not in df.columns:
                    continue

                # Only use broader windows:
                # broad window must be >= local window in both time and space
                # and not exactly the same window
                if by < ly or br < lr:
                    continue
                if by == ly and br == lr:
                    continue

                feat_name = f"hotspot_grad_{ly}y_{lr}m__{by}y_{br}m"
                inv_name = f"hotspot_grad_inv_{ly}y_{lr}m__{by}y_{br}m"

                df[feat_name] = (1 + df[local_col]) / (1 + df[broad_col])
                df[inv_name] = (1 + df[broad_col]) / (1 + df[local_col])

                created_cols.append(feat_name)
                created_cols.append(inv_name)

# ----------------------------
# Save result
# ----------------------------
out_file = "accidents_with_hotspot_gradients.csv"
df.to_csv(out_file, index=False)

print(f"Done. Created {len(created_cols)} new feature columns.")
print(f"Saved to: {out_file}")

files.download(out_file)

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

# ----------------------------
# Upload and read CSV
# ----------------------------

csv_file = "accidents_with_hotspot_gradients.csv"

df = pd.read_csv(csv_file, low_memory=False)

# ----------------------------
# Ensure required columns exist
# ----------------------------
required_cols = [
    "past_accidents_200m_3y",
    "exposure_avg_cyclists"
]

for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}")

# ----------------------------
# Convert to numeric safely
# ----------------------------
df["past_accidents_200m_3y"] = pd.to_numeric(df["past_accidents_200m_3y"], errors="coerce").fillna(0)
df["exposure_avg_cyclists"] = pd.to_numeric(df["exposure_avg_cyclists"], errors="coerce").fillna(0)

# ----------------------------
# Compute exposure-adjusted metric
# ----------------------------
df["exposure_adjusted_collision_history"] = (
    np.log1p(df["past_accidents_200m_3y"]) /
    np.log1p(df["exposure_avg_cyclists"] + 1)
)

# ----------------------------
# Save output
# ----------------------------
out_file = "accidents_with_exposure_adjusted_feature.csv"
df.to_csv(out_file, index=False)



In [ ]:
import pandas as pd
from google.colab import files

input_file = "output_with_speed_energy.csv"
output_file = "output_reordered.csv"

df = pd.read_csv(input_file, low_memory=False)

col = "casualty_severity"

if col not in df.columns:
    raise ValueError(f"Column '{col}' not found")

# Reorder explicitly
new_order = df.columns.drop(col).tolist() + [col]
df = df.reindex(columns=new_order)

# Verify
print("Last column:", df.columns[-1])

df.to_csv(output_file, index=False)
files.download(output_file)

In [ ]:
import pandas as pd
from google.colab import files

# ----------------------------
# Config
# ----------------------------
input_file = "accidents_with_exposure_adjusted_feature.csv"
output_file = "output_prior_severe.csv"

years = [1, 2, 3, 4, 5]
radii = [50, 100, 200, 300, 400, 500]

# ----------------------------
# Load data
# ----------------------------
df = pd.read_csv(input_file, low_memory=False)

created_cols = []
missing_cols = []

# ----------------------------
# Create prior severe-collision count:
# prior_severe_{r}m_{y}y = serious + fatal
# ----------------------------
for y in years:
    for r in radii:
        serious_col = f"past_accidents_{r}m_{y}y_serious"
        fatal_col = f"past_accidents_{r}m_{y}y_fatal"
        out_col = f"prior_severe_{r}m_{y}y"

        if serious_col in df.columns and fatal_col in df.columns:
            df[serious_col] = pd.to_numeric(df[serious_col], errors="coerce").fillna(0)
            df[fatal_col] = pd.to_numeric(df[fatal_col], errors="coerce").fillna(0)

            df[out_col] = df[serious_col] + df[fatal_col]
            created_cols.append(out_col)
        else:
            missing_cols.append((out_col, serious_col, fatal_col))

# ----------------------------
# Save
# ----------------------------
df.to_csv(output_file, index=False)

print(f"Created {len(created_cols)} prior severe features.")
print("First 10 created columns:")
for c in created_cols[:10]:
    print(" -", c)

if missing_cols:
    print(f"\nMissing {len(missing_cols)} combinations:")
    for out_col, serious_col, fatal_col in missing_cols[:10]:
        print(f" - {out_col} (missing {serious_col} or {fatal_col})")

files.download(output_file)

In [ ]:
import pandas as pd
from google.colab import files

# ----------------------------
# Load data
# ----------------------------
df = pd.read_csv("your_file_with_prior_severe.csv", low_memory=False)

# ----------------------------
# Ensure numeric
# ----------------------------
df["speedlimit_clean"] = pd.to_numeric(df["speedlimit_clean"], errors="coerce")

# ----------------------------
# Feature engineering
# ----------------------------
df["speed_energy_proxy"] = (df["speedlimit_clean"] / 20) ** 2

# ----------------------------
# Optional: audit missing values
# ----------------------------
missing = df["speed_energy_proxy"].isna().sum()
print(f"Missing values in speed_energy_proxy: {missing}")

# ----------------------------
# Save
# ----------------------------
output_file = "output_with_speed_energy.csv"
df.to_csv(output_file, index=False)

files.download(output_file)

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

# ----------------------------
# Config
# ----------------------------
input_file = "your_file_prior_severe_cleaned_2016.csv"
output_file = "your_file_prior_severe_cleaned_2017.csv"

year_col = "year"  # change if your year column has a different name

# ----------------------------
# Load
# ----------------------------
df = pd.read_csv(input_file, low_memory=False)

if year_col not in df.columns:
    raise ValueError(f"Column '{year_col}' not found.")

# ----------------------------
# Find target columns
# ----------------------------
target_cols = [
    c for c in df.columns
    if c.startswith("prior_severe_")
    and (c.endswith("_5y"))
]

print(f"Found {len(target_cols)} matching prior_severe columns.")

# ----------------------------
# Apply rule
# If year == 2014 and value == 0.0 -> set to NaN
# ----------------------------
audit = []

for col in target_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

    mask = (df[year_col] == 2017) & (df[col] == 0)

    changed = int(mask.sum())

    if changed > 0:
        audit.append((col, changed))
        df.loc[mask, col] = np.nan

# ----------------------------
# Audit
# ----------------------------
total_changed = sum(n for _, n in audit)

print(f"\nTotal cells changed: {total_changed:,}")

for col, changed in audit:
    print(f"{col}: {changed:,} values changed")

# ----------------------------
# Save
# ----------------------------
df.to_csv(output_file, index=False)

print(f"\nSaved: {output_file}")


In [ ]:
import pandas as pd
from google.colab import files

# ----------------------------
# Config
# ----------------------------
input_file = "accidents_with_exposure_adjusted_feature.csv"
output_file = "accidents_minutes_30_or_more.csv"

timestamp_col = "accident_time_utc"  # change if needed

# ----------------------------
# Load data
# ----------------------------
df = pd.read_csv(input_file, low_memory=False)

# ----------------------------
# Parse timestamp
# ----------------------------
df[timestamp_col] = pd.to_datetime(
    df[timestamp_col],
    errors="coerce",
    utc=True
)

valid_rows = df[timestamp_col].notna().sum()

# ----------------------------
# Filter minute >= 30
# ----------------------------
mask = df[timestamp_col].dt.minute >= 30
subset = df[mask].copy()

# ----------------------------
# Audit
# ----------------------------
matched_rows = len(subset)
total_rows = len(df)

pct_total = 100 * matched_rows / total_rows
pct_valid = 100 * matched_rows / valid_rows if valid_rows > 0 else 0

print(f"Total rows:               {total_rows:,}")
print(f"Valid timestamps:         {valid_rows:,}")
print(f"Rows with minute >= 30:   {matched_rows:,}")
print(f"% of all rows:            {pct_total:.2f}%")
print(f"% of valid timestamps:    {pct_valid:.2f}%")

# ----------------------------
# Save subset
# ----------------------------
subset.to_csv(output_file, index=False)

print(f"\nSaved: {output_file}")


In [ ]:
import pandas as pd

csv_file = "accidents_with_exposure_adjusted_feature.csv"  # change if needed

# Load CSV
df = pd.read_csv(csv_file)

# Keep only rows with valid coordinates
coords = df[["latitude", "longitude"]].dropna()

# Count unique pairs
unique_pairs = coords.drop_duplicates()

total_rows = len(df)
valid_rows = len(coords)
num_unique = len(unique_pairs)
num_duplicates = valid_rows - num_unique

print(f"Total rows: {total_rows:,}")
print(f"Rows with valid coordinates: {valid_rows:,}")
print(f"Unique (lat, lng) pairs: {num_unique:,}")
print(f"Duplicate coordinate pairs: {num_duplicates:,}")

In [ ]:
# process jsonl with past hour
#!/usr/bin/env python3
import requests
import json
import os, time
import pandas as pd

INPUT_JSONL = "https://YOUR_STORAGE_ACCOUNT.blob.core.windows.net/london/open_meteo_data/open_meteo_archive_and_forecast_full_4.jsonl?sp=r&st=2026-06-29T13:04:43Z&se=2026-06-29T21:19:43Z&spr=https&sv=2026-02-06&sr=b&sig=YOUR_AZURE_SAS_SIGNATURE"
OUTPUT_CSV = "archive_4.csv"

# ---------------------------------------------------------------------
# OUTPUT SCHEMA
# - hourly lags are exact offsets from the matched hour
# - last_3h / last_6h windows INCLUDE the matched hour
# - hourly match is the nearest hourly timestamp in the payload
# ---------------------------------------------------------------------

PROVENANCE_COLUMNS = [
    "_row_index",
    "accident_datetime_utc",
    "source_latitude",
    "source_longitude",
    "archive_status",
    "archive_error",
    "archive_request_count",
]

HOURLY_OPS = [
    ("temperature_2m", "avg"),
    ("relativehumidity_2m", "avg"),
    ("dewpoint_2m", "avg"),
    ("apparent_temperature", "avg"),
    ("precipitation", "sum"),
    ("rain", "sum"),
    ("snowfall", "sum"),
    ("snow_depth", "sum"),
    ("weathercode", "avg"),
    ("pressure_msl", "avg"),
    ("surface_pressure", "avg"),
    ("cloudcover", "avg"),
    ("cloudcover_low", "avg"),
    ("cloudcover_mid", "avg"),
    ("cloudcover_high", "avg"),
    ("windspeed_10m", "avg"),
    ("winddirection_10m", "avg"),
    ("windgusts_10m", "avg"),
    ("et0_fao_evapotranspiration", "avg"),
    ("vapour_pressure_deficit", "avg"),
    ("windspeed_100m", "avg"),
    ("winddirection_100m", "avg"),
    ("soil_temperature_0_to_7cm", "avg"),
    ("soil_temperature_7_to_28cm", "avg"),
    ("soil_temperature_28_to_100cm", "avg"),
    ("soil_temperature_100_to_255cm", "avg"),
    ("soil_moisture_0_to_7cm", "avg"),
    ("soil_moisture_7_to_28cm", "avg"),
    ("soil_moisture_28_to_100cm", "avg"),
    ("soil_moisture_100_to_255cm", "avg"),
]

DAILY_VARS = [
    "weathercode",
    "temperature_2m_max",
    "temperature_2m_min",
    "apparent_temperature_max",
    "apparent_temperature_min",
    "sunrise",
    "sunset",
    "daylight_duration",
    "sunshine_duration",
    "precipitation_sum",
    "rain_sum",
    "snowfall_sum",
    "precipitation_hours",
    "windspeed_10m_max",
    "windgusts_10m_max",
    "winddirection_10m_dominant",
    "shortwave_radiation_sum",
    "et0_fao_evapotranspiration",
]

FEATURE_COLUMNS = [
    "archive_retrieved_latitude",
    "archive_retrieved_longitude",
    "archive_hour_match_gap_seconds",
    "weather_datetime_utc",
]

for var, op in HOURLY_OPS:
    FEATURE_COLUMNS.extend([
        f"archive.hourly.{var}",
        f"archive.hourly.{var}_lag_1h",
        f"archive.hourly.{var}_lag_3h",
    ])
    if op == "avg":
        FEATURE_COLUMNS.extend([
            f"archive.hourly.{var}_avg_last_3h",
            f"archive.hourly.{var}_avg_last_6h",
        ])
    else:
        FEATURE_COLUMNS.extend([
            f"archive.hourly.{var}_sum_last_3h",
            f"archive.hourly.{var}_sum_last_6h",
        ])

for var in DAILY_VARS:
    FEATURE_COLUMNS.append(f"archive.daily.{var}")

OUTPUT_COLUMNS = PROVENANCE_COLUMNS + FEATURE_COLUMNS


def to_ts(x):
    ts = pd.to_datetime(x, utc=True, errors="coerce")
    return None if pd.isna(ts) else ts


def to_num(x):
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    try:
        return float(x)
    except Exception:
        return None


def mean_or_none(values):
    vals = [to_num(v) for v in values]
    vals = [v for v in vals if v is not None]
    return None if not vals else sum(vals) / len(vals)


def sum_or_none(values):
    vals = [to_num(v) for v in values]
    vals = [v for v in vals if v is not None]
    return None if not vals else sum(vals)


def build_hourly_index(hourly):
    raw_times = hourly.get("time", [])
    times = [to_ts(t) for t in raw_times]
    times = [t for t in times if t is not None]
    time_to_idx = {t.isoformat(): i for i, t in enumerate(times)}
    return times, time_to_idx


def nearest_hour_index(times, accident_ts):
    """
    Return the latest hourly timestamp that is <= accident_ts.
    Never uses future observations.
    """
    if not times or accident_ts is None:
        return None, None

    idx = None
    for i, t in enumerate(times):
        if t <= accident_ts:
            idx = i
        else:
            break

    if idx is None:
        return None, None

    gap_seconds = int((accident_ts - times[idx]).total_seconds())
    return idx, gap_seconds


def exact_hour_index(time_to_idx, ts):
    if ts is None:
        return None
    return time_to_idx.get(ts.isoformat())


def series_value(values, idx):
    if idx is None:
        return None
    if idx < 0 or idx >= len(values):
        return None
    return values[idx]


def window_values(hourly, time_to_idx, center_ts, var, hours_back):
    """
    Window includes center_ts and the prior hours_back-1 hours.
    For example:
      hours_back=3 -> [t-2h, t-1h, t]
      hours_back=6 -> [t-5h, t-4h, t-3h, t-2h, t-1h, t]
    """
    vals = []
    values = hourly.get(var, [])

    for offset in range(hours_back):
        ts = center_ts - pd.Timedelta(hours=offset)
        idx = exact_hour_index(time_to_idx, ts)
        v = series_value(values, idx)
        if v is not None and not pd.isna(v):
            vals.append(v)

    return vals


def build_row(obj):
    row = {k: obj.get(k) for k in PROVENANCE_COLUMNS}

    accident_ts = to_ts(obj.get("accident_datetime_utc"))
    archive = obj.get("archive") or {}
    hourly = archive.get("hourly") or {}
    daily = archive.get("daily") or {}

    row["archive_retrieved_latitude"] = archive.get("latitude")
    row["archive_retrieved_longitude"] = archive.get("longitude")

    times, time_to_idx = build_hourly_index(hourly)
    nearest_idx, gap_seconds = nearest_hour_index(times, accident_ts)
    row["archive_hour_match_gap_seconds"] = gap_seconds

    matched_hour_ts = times[nearest_idx] if nearest_idx is not None else None
    row["weather_datetime_utc"] = (
        matched_hour_ts.isoformat() if matched_hour_ts is not None else None
    )

    # Hourly features
    for var, op in HOURLY_OPS:
        values = hourly.get(var, [])
        matched_value = series_value(values, nearest_idx)

        row[f"archive.hourly.{var}"] = matched_value

        lag_1h_ts = matched_hour_ts - pd.Timedelta(hours=1) if matched_hour_ts is not None else None
        lag_3h_ts = matched_hour_ts - pd.Timedelta(hours=3) if matched_hour_ts is not None else None

        row[f"archive.hourly.{var}_lag_1h"] = series_value(values, exact_hour_index(time_to_idx, lag_1h_ts))
        row[f"archive.hourly.{var}_lag_3h"] = series_value(values, exact_hour_index(time_to_idx, lag_3h_ts))

        if matched_hour_ts is not None:
            window_3h = window_values(hourly, time_to_idx, matched_hour_ts, var, 3)
            window_6h = window_values(hourly, time_to_idx, matched_hour_ts, var, 6)
        else:
            window_3h = []
            window_6h = []

        if op == "avg":
            row[f"archive.hourly.{var}_avg_last_3h"] = mean_or_none(window_3h)
            row[f"archive.hourly.{var}_avg_last_6h"] = mean_or_none(window_6h)
        else:
            row[f"archive.hourly.{var}_sum_last_3h"] = sum_or_none(window_3h)
            row[f"archive.hourly.{var}_sum_last_6h"] = sum_or_none(window_6h)

    # Daily features: exact date match on the daily time series
    daily_times = [to_ts(t) for t in daily.get("time", [])]
    daily_times = [t for t in daily_times if t is not None]
    daily_index = {t.date().isoformat(): i for i, t in enumerate(daily_times)}

    accident_date = accident_ts.date().isoformat() if accident_ts is not None else None
    daily_match_idx = daily_index.get(accident_date)

    for var in DAILY_VARS:
        values = daily.get(var, [])
        row[f"archive.daily.{var}"] = series_value(values, daily_match_idx)

    return row


def main():
    rows = []

    response = requests.get(INPUT_JSONL, stream=True)
    response.raise_for_status()
    print("Start parsing")

    start = time.time()

    for i, line in enumerate(response.iter_lines(decode_unicode=True), start=1):
        if i == 10:
            print("Received first line")
        if not line:
            continue

        obj = json.loads(line)
        if i == 10:
            print("Parsed first JSON")
        rows.append(build_row(obj))
        if i == 10:
            print("Built first row")

        if i % 100 == 0:
            elapsed = time.time() - start
            print(
                f"Processed {i:,} rows "
                f"({i/elapsed:,.0f} rows/s, {elapsed/60:.1f} min elapsed)"
            )

    df = pd.DataFrame(rows)

    # Enforce column order and keep any unexpected extras at the end if they exist
    for col in OUTPUT_COLUMNS:
        if col not in df.columns:
            df[col] = None

    extras = [c for c in df.columns if c not in OUTPUT_COLUMNS]
    df = df[OUTPUT_COLUMNS + extras]

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved {len(df):,} rows -> {OUTPUT_CSV}")
    print(f"Columns: {len(df.columns)}")


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd

INPUT_CSV = "accidents_final_processed.csv"
OUTPUT_CSV = "accidents_final_processed_no_archive.csv"
LOG_FILE = "dropped_columns.txt"

# Load CSV
df = pd.read_csv(INPUT_CSV)

# Identify archive columns
archive_cols = [c for c in df.columns if c.startswith("archive.")]

# Log them
with open(LOG_FILE, "w") as f:
    f.write(f"Dropped {len(archive_cols)} columns:\n")
    for c in archive_cols:
        f.write(c + "\n")

# Drop them
df_clean = df.drop(columns=archive_cols)

# Save result
df_clean.to_csv(OUTPUT_CSV, index=False)

print(f"Dropped {len(archive_cols)} columns")
print(f"Saved cleaned CSV -> {OUTPUT_CSV}")
print(f"Log -> {LOG_FILE}")

In [ ]:
import pandas as pd
from google.colab import files

# Upload files

file1 = "accidents_final_processed.csv"
file2 = "open_meteo_archive_final_utc.csv"

# Read headers only
df1 = pd.read_csv(file1, nrows=0)
df2 = pd.read_csv(file2, nrows=0)

cols1 = df1.columns
cols2 = set(df2.columns)

prefix = "archive.hourly."

missing = []

for col in cols1:
    if col.startswith(prefix):
        base = col[len(prefix):]  # remove prefix only

        # direct match only (no translation, no mapping)
        if base not in cols2:
            missing.append(col)

print(f"Missing columns: {len(missing)}\n")
for c in missing:
    print(c)

In [ ]:
import pandas as pd

INPUT_CSV = "open_meteo_archive_final_utc.csv"

df = pd.read_csv(INPUT_CSV)

# --- normalize weather_time to UTC ISO format ---
df["weather_time"] = pd.to_datetime(df["weather_time"], utc=True, errors="coerce")
df["accident_time_utc"] = pd.to_datetime(df["accident_time_utc"], utc=True, errors="coerce")

# --- check ordering ---
bad = df[df["weather_time"] > df["accident_time_utc"]]

print(f"Total rows: {len(df)}")
print(f"Invalid rows (weather_time > accident_time_utc): {len(bad)}")

if len(bad) > 0:
    print("\nExamples:")
    print(bad[["weather_time", "accident_time_utc"]].head(20))

# --- (optional) save cleaned version ---
OUTPUT_CSV = "open_meteo_archive_final_utc_checked.csv"
df.to_csv(OUTPUT_CSV, index=False)

print(f"\nSaved -> {OUTPUT_CSV}")

In [ ]:
import pandas as pd

FILE1 = "accidents_final_processed.csv"
FILE2 = "open_meteo_archive_final_utc.csv"
OUTPUT = "merged_corrected.csv"

df1 = pd.read_csv(FILE1)
df2 = pd.read_csv(FILE2)

# normalize timestamps
df1["accident_time_utc"] = pd.to_datetime(df1["accident_time_utc"], utc=True, errors="coerce")
df2["accident_time_utc"] = pd.to_datetime(df2["accident_time_utc"], utc=True, errors="coerce")

# build lookup using corrected keys from file2
df2_indexed = df2.set_index(["source_latitude", "source_longitude", "accident_time_utc"])

archive_cols = [c for c in df1.columns if c.startswith("archive.hourly.")]

replacements_log = []

for i, row in df1.iterrows():

    key = (
        row.get("latitude"),
        row.get("longitude"),
        row.get("accident_time_utc")
    )

    if key not in df2_indexed.index:
        continue

    match = df2_indexed.loc[key]

    # handle single vs multiple matches
    if isinstance(match, pd.DataFrame):
        matches = match.to_dict("records")
    else:
        matches = [match]

    for m in matches:

        # --- replace coordinates ---
        old_lat = row["archive_retrieved_latitude"]
        old_lon = row["archive_retrieved_longitude"]

        new_lat = key[0]
        new_lon = key[1]

        df1.at[i, "archive_retrieved_latitude"] = new_lat
        df1.at[i, "archive_retrieved_longitude"] = new_lon

        # --- copy weather_time ---
        new_weather_time = m.get("weather_time")

        df1.at[i, "weather_time"] = new_weather_time

        replacements_log.append({
            "row": i,
            "field": "weather_time",
            "from": None,
            "to": new_weather_time
        })

        replacements_log.append({
            "row": i,
            "field": "coords",
            "from": (old_lat, old_lon),
            "to": (new_lat, new_lon)
        })

        # --- replace hourly features ---
        for col in archive_cols:
            raw_col = col.replace("archive.hourly.", "")

            if raw_col in df2.columns:
                old_val = df1.at[i, col]
                new_val = m.get(raw_col)

                df1.at[i, col] = new_val

                replacements_log.append({
                    "row": i,
                    "field": col,
                    "from": old_val,
                    "to": new_val
                })

df1.to_csv(OUTPUT, index=False)

pd.DataFrame(replacements_log).to_csv("replacement_log.csv", index=False)

print(f"Saved -> {OUTPUT}")
print(f"Replacements -> {len(replacements_log)}")
print("Log -> replacement_log.csv")

In [ ]:
import pandas as pd

INPUT_CSV = "merged_corrected.csv"
OUTPUT_CSV = "merged_corrected_with_gap.csv"

df = pd.read_csv(INPUT_CSV, low_memory=False)

# Parse timestamps
df["accident_time_utc"] = pd.to_datetime(
    df["accident_time_utc"],
    utc=True,
    errors="coerce"
)

df["weather_time"] = pd.to_datetime(
    df["weather_time"],
    utc=True,
    errors="coerce"
)

# Compute gap in seconds
df["archive_hour_match_gap_seconds"] = (
    df["accident_time_utc"] - df["weather_time"]
).dt.total_seconds().astype("Int64")

# Check for invalid rows (weather after accident)
invalid = df[df["archive_hour_match_gap_seconds"] < 0]

print(f"Invalid rows (weather_time > accident_time_utc): {len(invalid)}")

if len(invalid):
    print(invalid[
        [
            "accident_time_utc",
            "weather_time",
            "archive_hour_match_gap_seconds"
        ]
    ].head(20))

df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved -> {OUTPUT_CSV}")

In [ ]:
import pandas as pd

INPUT_CSV = "merged_corrected_with_gap.csv"
OUTPUT_CSV = "output.csv"

df = pd.read_csv(INPUT_CSV, low_memory=False)

affected = {}

for col in df.columns:
    n = (df[col] == "Not applicable").sum()
    if n > 0:
        affected[col] = int(n)

# Replace
df.replace("Not applicable", "", inplace=True)

df.to_csv(OUTPUT_CSV, index=False)

print(f"Total replacements: {sum(affected.values())}")
print(f"Columns affected: {len(affected)}\n")

for col, n in affected.items():
    print(f"{col}: {n}")

print(f"\nSaved -> {OUTPUT_CSV}")

In [ ]:
import re
import pandas as pd

INPUT_CSV = "merged_corrected.csv"
OUTPUT_CSV = "output.csv"

df = pd.read_csv(INPUT_CSV, low_memory=False)

renamed = {}

for col in df.columns:
    m = re.fullmatch(r"n_pubs_(\d+)m", col)
    if m:
        new_col = f"n_pubs_crow_{m.group(1)}m"
        renamed[col] = new_col

df.rename(columns=renamed, inplace=True)

print(f"Renamed {len(renamed)} columns:\n")
for old, new in renamed.items():
    print(f"{old} -> {new}")

df.to_csv(OUTPUT_CSV, index=False)

print(f"\nSaved -> {OUTPUT_CSV}")

In [ ]:
import pandas as pd
from pandas.api.types import is_integer_dtype

INPUT_CSV = "merged_corrected.csv"

df = pd.read_csv(INPUT_CSV, low_memory=False)

int_cols = [col for col in df.columns if is_integer_dtype(df[col])]

print(f"Integer columns: {len(int_cols)}\n")

for col in int_cols:
    print(f"{col} ({df[col].dtype})")

In [ ]:
import pandas as pd

INPUT_CSV = "merged_corrected.csv"
OUTPUT_CSV = "output.csv"

df = pd.read_csv(INPUT_CSV, low_memory=False)

cols_to_float = [
    "archive.hourly.relativehumidity_2m",
    "archive.hourly.cloudcover",
    "archive.hourly.cloudcover_low",
    "archive.hourly.cloudcover_mid",
    "archive.hourly.cloudcover_high",
    "archive.hourly.winddirection_10m",
    "archive.hourly.winddirection_100m",
]

converted = []

for col in cols_to_float:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype(float).round(1)
        converted.append(col)

df.to_csv(OUTPUT_CSV, index=False)

print(f"Converted {len(converted)} columns to float (1 decimal):")
for c in converted:
    print("-", c)

print(f"\nSaved -> {OUTPUT_CSV}")

In [ ]:
import pandas as pd

INPUT_CSV = "output_28.csv"
OUTPUT_CSV = "output_29.csv"

df = pd.read_csv(INPUT_CSV, low_memory=False)

# Columns to process
target_cols = [c for c in df.columns if c.startswith("past_accidents_100m_5y")]

total_changed = 0

for col in target_cols:
    mask = (
        (df["year"] == 2016) &
        (pd.to_numeric(df[col], errors="coerce").isin([0, 0.0]))
    )

    changed = mask.sum()
    if changed:
        df.loc[mask, col] = ""
        total_changed += changed
        print(f"{col}: cleared {changed} cells")

df.to_csv(OUTPUT_CSV, index=False)

print(f"\nTotal cells cleared: {total_changed}")
print(f"Saved -> {OUTPUT_CSV}")

In [ ]:
import pandas as pd

INPUT_CSV = "input_nn_final.csv"
OUTPUT_CSV = "output_dropped.csv"

cols_to_drop = [
    "archive.hourly.winddirection_10m_avg_last_3h",
    "archive.hourly.winddirection_10m_avg_last_6h",
    "archive.hourly.winddirection_100m_avg_last_3h",
    "archive.hourly.winddirection_100m_avg_last_6h",
]

df = pd.read_csv(INPUT_CSV, low_memory=False)

existing = [c for c in cols_to_drop if c in df.columns]
missing = [c for c in cols_to_drop if c not in df.columns]

df.drop(columns=existing, inplace=True)

df.to_csv(OUTPUT_CSV, index=False)

print("Dropped columns:")
for c in existing:
    print(f"- {c}")

if missing:
    print("\nColumns not found:")
    for c in missing:
        print(f"- {c}")

print(f"\nSaved -> {OUTPUT_CSV}")

In [ ]:
import pandas as pd
import numpy as np

from math import sin, cos, radians

FILE1 = "accidents_final_processed.csv"
FILE2 = "open_meteo_archive_final_utc.csv"
OUTPUT = "merged_corrected.csv"

df1 = pd.read_csv(FILE1)
df2 = pd.read_csv(FILE2)

df1["accident_time_utc"] = pd.to_datetime(df1["accident_time_utc"], utc=True, errors="coerce")
df2["accident_time_utc"] = pd.to_datetime(df2["accident_time_utc"], utc=True, errors="coerce")

df2_indexed = df2.set_index(["source_latitude", "source_longitude", "accident_time_utc"])

archive_cols = [c for c in df1.columns if c.startswith("archive.hourly.")]

replacements_log = []


def to_sincos(x):
    if pd.isna(x):
        return None, None
    r = radians(float(x))
    return sin(r), cos(r)


def avg(vals):
    vals = [v for v in vals if v is not None and not pd.isna(v)]
    return sum(vals) / len(vals) if vals else None


for i, row in df1.iterrows():

    key = (
        row.get("latitude"),
        row.get("longitude"),
        row.get("accident_time_utc")
    )

    if key not in df2_indexed.index:
        continue

    match = df2_indexed.loc[key]

    if isinstance(match, pd.DataFrame):
        matches = match.to_dict("records")
    else:
        matches = [match]

    for m in matches:

        # -------------------------
        # coords
        # -------------------------
        old_lat = row["archive_retrieved_latitude"]
        old_lon = row["archive_retrieved_longitude"]

        new_lat = key[0]
        new_lon = key[1]

        df1.at[i, "archive_retrieved_latitude"] = new_lat
        df1.at[i, "archive_retrieved_longitude"] = new_lon


        # -------------------------
        # NEW: wind sin/cos features
        # -------------------------
        for var in ["winddirection_10m", "winddirection_100m"]:

            base = m.get(var)
            s, c = to_sincos(base)

            df1.at[i, f"archive.hourly.{var}_sin"] = s
            df1.at[i, f"archive.hourly.{var}_cos"] = c

            # rolling windows (from already overwritten df2 values)
            sin3, cos3 = [], []
            sin6, cos6 = [], []

            for j in range(3):
                v = m.get(var)  # same row proxy; simplified minimal assumption
                s, c = to_sincos(v)
                sin3.append(s)
                cos3.append(c)

            for j in range(6):
                v = m.get(var)
                s, c = to_sincos(v)
                sin6.append(s)
                cos6.append(c)

            df1.at[i, f"archive.hourly.{var}_sin_avg_last_3h"] = avg(sin3)
            df1.at[i, f"archive.hourly.{var}_cos_avg_last_3h"] = avg(cos3)
            df1.at[i, f"archive.hourly.{var}_sin_avg_last_6h"] = avg(sin6)
            df1.at[i, f"archive.hourly.{var}_cos_avg_last_6h"] = avg(cos6)

        # -------------------------
        # logging
        # -------------------------
        replacements_log.append({
            "row": i,
            "field": "wind_sincos_added",
            "from": None,
            "to": "computed"
        })

df1.to_csv(OUTPUT, index=False)
pd.DataFrame(replacements_log).to_csv("replacement_log.csv", index=False)

print(f"Saved -> {OUTPUT}")
print(f"Replacements -> {len(replacements_log)}")

In [ ]:
import pandas as pd

FILE1 = "input_nn_final.csv"
FILE2 = "open_meteo_archive_final_utc.csv"
OUTPUT = "merged_corrected.csv"

df1 = pd.read_csv(FILE1)
df2 = pd.read_csv(FILE2)

# normalize timestamps
df1["accident_time_utc"] = pd.to_datetime(df1["accident_time_utc"], utc=True, errors="coerce")
df2["accident_time_utc"] = pd.to_datetime(df2["accident_time_utc"], utc=True, errors="coerce")

# build lookup using corrected keys from file2
df2_indexed = df2.set_index(["source_latitude", "source_longitude", "accident_time_utc"])

weather_cols = [
    "winddirection_10m_sin",
    "winddirection_10m_cos",
    "winddirection_10m_sin_avg_last_3h",
    "winddirection_10m_cos_avg_last_3h",
    "winddirection_10m_dir_avg_last_3h",

    "winddirection_100m_sin",
    "winddirection_100m_cos",
    "winddirection_100m_sin_avg_last_6h",
    "winddirection_100m_cos_avg_last_6h",
    "winddirection_100m_dir_avg_last_6h",
]

replacements_log = []

for i, row in df1.iterrows():

    key = (
        row.get("latitude"),
        row.get("longitude"),
        row.get("accident_time_utc")
    )

    if key not in df2_indexed.index:
        continue

    match = df2_indexed.loc[key]

    # handle single vs multiple matches
    if isinstance(match, pd.DataFrame):
        matches = match.to_dict("records")
    else:
        matches = [match]

    for m in matches:

        # --- replace coordinates ---
        old_lat = row["archive_retrieved_latitude"]
        old_lon = row["archive_retrieved_longitude"]

        new_lat = key[0]
        new_lon = key[1]

        df1.at[i, "archive_retrieved_latitude"] = new_lat
        df1.at[i, "archive_retrieved_longitude"] = new_lon

        # --- copy weather_time ---
        new_weather_time = m.get("weather_time")

        df1.at[i, "weather_time"] = new_weather_time

        replacements_log.append({
            "row": i,
            "field": "weather_time",
            "from": None,
            "to": new_weather_time
        })

        replacements_log.append({
            "row": i,
            "field": "coords",
            "from": (old_lat, old_lon),
            "to": (new_lat, new_lon)
        })

        # --- replace hourly features ---
        for col in weather_cols:
            if col in df2.columns:
                df1.at[i, col] = m.get(col)

df1.to_csv(OUTPUT, index=False)

pd.DataFrame(replacements_log).to_csv("replacement_log.csv", index=False)

print(f"Saved -> {OUTPUT}")
print(f"Replacements -> {len(replacements_log)}")
print("Log -> replacement_log.csv")

In [ ]:
import pandas as pd

FILE = "input_nn_final_sin_cos_rounded_1.csv"

df = pd.read_csv(FILE)

for col in df.columns:
    if not pd.api.types.is_numeric_dtype(df[col]):
        continue

    # only float columns
    if not pd.api.types.is_float_dtype(df[col]):
        continue

    values = df[col].dropna()

    has_more_than_2 = values.astype(str).str.contains(r"\.\d{3,}$").any()

    if has_more_than_2:
        print(col)

In [ ]:
import pandas as pd

INPUT = "input_nn_final_sin_cos_rounded_1.csv"
OUTPUT = "input_nn_final_sin_cos_rounded_2.csv"

df = pd.read_csv(INPUT)

cols_to_round = [
    "archive.hourly.temperature_2m_avg_last_3h",
    "archive.hourly.temperature_2m_avg_last_6h",
    "archive.hourly.relativehumidity_2m_avg_last_3h",
    "archive.hourly.relativehumidity_2m_avg_last_6h",
    "archive.hourly.dewpoint_2m_avg_last_3h",
    "archive.hourly.dewpoint_2m_avg_last_6h",
    "archive.hourly.apparent_temperature_avg_last_3h",
    "archive.hourly.apparent_temperature_avg_last_6h",
    "archive.hourly.pressure_msl_avg_last_3h",
    "archive.hourly.pressure_msl_avg_last_6h",
    "archive.hourly.surface_pressure_avg_last_3h",
    "archive.hourly.surface_pressure_avg_last_6h",
    "archive.hourly.cloudcover_avg_last_3h",
    "archive.hourly.cloudcover_avg_last_6h",
    "archive.hourly.cloudcover_low_avg_last_3h",
    "archive.hourly.cloudcover_low_avg_last_6h",
    "archive.hourly.cloudcover_mid_avg_last_3h",
    "archive.hourly.cloudcover_mid_avg_last_6h",
    "archive.hourly.cloudcover_high_avg_last_3h",
    "archive.hourly.cloudcover_high_avg_last_6h",
    "archive.hourly.windspeed_10m_avg_last_3h",
    "archive.hourly.windspeed_10m_avg_last_6h",
    "archive.hourly.windgusts_10m_avg_last_3h",
    "archive.hourly.windgusts_10m_avg_last_6h",
    "archive.hourly.et0_fao_evapotranspiration_avg_last_3h",
    "archive.hourly.et0_fao_evapotranspiration_avg_last_6h",
    "archive.hourly.vapour_pressure_deficit_avg_last_3h",
    "archive.hourly.vapour_pressure_deficit_avg_last_6h",
    "archive.hourly.windspeed_100m_avg_last_3h",
    "archive.hourly.windspeed_100m_avg_last_6h",
    "archive.hourly.soil_temperature_0_to_7cm_avg_last_3h",
    "archive.hourly.soil_temperature_0_to_7cm_avg_last_6h",
    "archive.hourly.soil_temperature_7_to_28cm_avg_last_3h",
    "archive.hourly.soil_temperature_7_to_28cm_avg_last_6h",
    "archive.hourly.soil_temperature_28_to_100cm_avg_last_3h",
    "archive.hourly.soil_temperature_28_to_100cm_avg_last_6h",
    "archive.hourly.soil_temperature_100_to_255cm_avg_last_3h",
    "archive.hourly.soil_temperature_100_to_255cm_avg_last_6h",
    "archive.hourly.soil_moisture_0_to_7cm",
    "archive.hourly.soil_moisture_0_to_7cm_lag_1h",
    "archive.hourly.soil_moisture_0_to_7cm_lag_3h",
    "archive.hourly.soil_moisture_0_to_7cm_avg_last_3h",
    "archive.hourly.soil_moisture_0_to_7cm_avg_last_6h",
    "archive.hourly.soil_moisture_7_to_28cm",
    "archive.hourly.soil_moisture_7_to_28cm_lag_1h",
    "archive.hourly.soil_moisture_7_to_28cm_lag_3h",
    "archive.hourly.soil_moisture_7_to_28cm_avg_last_3h",
    "archive.hourly.soil_moisture_7_to_28cm_avg_last_6h",
    "archive.hourly.soil_moisture_28_to_100cm",
    "archive.hourly.soil_moisture_28_to_100cm_lag_1h",
    "archive.hourly.soil_moisture_28_to_100cm_lag_3h",
    "archive.hourly.soil_moisture_28_to_100cm_avg_last_3h",
    "archive.hourly.soil_moisture_28_to_100cm_avg_last_6h",
    "archive.hourly.soil_moisture_100_to_255cm",
    "archive.hourly.soil_moisture_100_to_255cm_lag_1h",
    "archive.hourly.soil_moisture_100_to_255cm_lag_3h",
    "archive.hourly.soil_moisture_100_to_255cm_avg_last_3h",
    "archive.hourly.soil_moisture_100_to_255cm_avg_last_6h",
    ]

for col in cols_to_round:
    if col in df.columns:
        df[col] = df[col].round(4)

df.to_csv(OUTPUT, index=False)

print(f"Saved -> {OUTPUT}")

In [ ]:
import pandas as pd

INPUT = "input_nn_final_sin_cos_rounded_2.csv"
OUTPUT = "input_nn_final_sin_cos_rounded_2_final.csv"

df = pd.read_csv(INPUT)

# Move column to the end
col = df.pop("casualty_severity")
df["casualty_severity"] = col

df.to_csv(OUTPUT, index=False)

print(f"Saved -> {OUTPUT}")

In [ ]:
import pandas as pd

# Path to your CSV file
input_file = "/content/accidents_final_processed.csv"

# Read the CSV
df = pd.read_csv(input_file)

# Select 10 random rows (or all rows if the file has fewer than 10)
sample_df = df.sample(n=min(10, len(df)), random_state=42)

# Save the sample to a new CSV
output_file = "/content/sample_10_rows.csv"
sample_df.to_csv(output_file, index=False)

print(f"Original shape: {df.shape}")
print(f"Sample shape: {sample_df.shape}")
print(f"Saved to: {output_file}")

# Display the sample
display(sample_df)

In [ ]:
import pandas as pd
from google.colab import files


accidents_file = "input_nn_final_sin_cos.csv"
holidays_file = "ukbankholidays-jul19.xls"

# Read the files
accidents = pd.read_csv(accidents_file)
# Read the holidays Excel file (no header)
holidays = pd.read_excel(holidays_file, header=None)

# Remove empty rows
holidays = holidays.dropna()

# Remove the title row ("UK BANK HOLIDAYS")
holidays = holidays[holidays[0] != "UK BANK HOLIDAYS"]

# Convert dates
holiday_dates = pd.to_datetime(
    holidays[0],
    format="%d-%b-%Y",
    errors="coerce"
).dt.date

# Remove any invalid dates
holiday_set = set(holiday_dates.dropna())

# Convert accident UTC timestamp to date
accident_dates = pd.to_datetime(
    accidents["accident_time_utc"],
    utc=True,
    errors="coerce"
).dt.date

# Create new column: 1 if bank holiday, else 0
bank_holiday = accident_dates.isin(holiday_set).astype(int)

# Insert before the last column
insert_position = len(accidents.columns) - 1
accidents.insert(insert_position, "bank_holiday", bank_holiday)

# Save the result
output_file = "accidents_with_bank_holiday.csv"
accidents.to_csv(output_file, index=False)

print(f"Saved to {output_file}")